# 07 -- Joint Hurdle BiLSTM

Главная идея -- не учить classifier и regressor как две независимые сети. Один shared encoder сразу оптимизирует итоговый прогноз в `log1p`-пространстве.

Финальный сабмит:

[lstm_hurdle_v4_robust.csv](/submissions/lstm_hurdle_v4_robust.csv)

**Public RMSLE: 1.6509102971.**

## Что изменилось относительно изначальной Hurdle BiLSTM

Изначальная сильная версия была такой:

```text
sequence -> BiLSTM -> last/mean/max
                         \
                          -> fusion -> classifier
static -> MLP            /

sequence -> BiLSTM -> last/mean/max
                         \
                          -> fusion -> positive regressor
static -> MLP            /
```

Classifier и regressor были двумя независимыми `HybridLSTM` с разными весами.

Финальный прогноз:

`pred_log = P(GMV > 0) * positive_log`.

Что поменял:

- две независимые сети -> один shared encoder;
- отдельный BCE-classifier и positive-only regressor -> три головы на общем representation;
- добавил `direct_head`, который сразу предсказывает итоговый `log1p(GMV)`;
- hurdle и direct смешиваются обучаемым весом `w`;
- главный loss считается по финальному `pred_log`, а не по двум независимым surrogate-задачам;
- `BatchNorm` перед sequence -> `LayerNorm` после projection;
- `Linear(96) + BiLSTM(hidden=128)` -> `Linear(80) + BiLSTM(hidden=112)`;
- pooling `last + mean + max` -> `last + masked mean + masked max + attention`;
- добавил recency bias в attention;
- left padding больше не проходит через LSTM как настоящая история -- использую `history_length`, mask и `pack_padded_sequence`;
- добавил отдельную short-summary ветку на окнах 1/3/7/14/30/60/90 дней;
- sequence выросла с 40 до 53 признаков на день;
- static preprocessing теперь фитится отдельно на train части каждого temporal fold;
- `user_id` embedding оставил optional, но в финальном run выключил;
- future-horizon calendar оставил optional, но в финальном run выключил;
- conv stem оставил optional, но в финальном run выключил;
- число эпох выбираю по 3 expanding temporal folds, а не по одному последнему cutoff;
- final prediction -- log-space ensemble двух seed;
- checkpoint сохраняю после каждой эпохи.

Главное изменение -- модель напрямую оптимизирует тот `pred_log`, который потом идет в RMSLE.

**Новая архитектура:**

Текущая архитектура:

```text
sequence (53/day)
    |
    -> Linear(80) -> LayerNorm -> GELU
    |
    -> 2-layer BiLSTM(hidden=112)
    |
    -> last / masked mean / masked max / attention
    |
    -> sequence head -> 144
                           \
short summary (93)          \
    |                        \
    -> MLP -> 64              \
                               -> concat -> fusion -> 96
static (235)                 /                  |
    |                       /                   |
    -> MLP -> 96           /                    |
                                              / | \
                                             /  |  \
                                            /   |   \
                                           /    |    \
                                     gate_head  |  direct_head
                                         |      |      |
                                    P(GMV > 0)  |  direct_log
                                                |
                                          positive_head 
                                                |
                                           positive_log
```

`hurdle_log = P(GMV > 0) * positive_log`

```text
                       hurdle_log
                           \
                            -> learned blend -> pred_log
                           /
                      direct_log
```

`pred_log = w * hurdle_log + (1 - w) * direct_log`

## Что поменялось в обучении этой версии

Старая CV была жестко ограничена `6` эпохами, и лучший mean temporal RMSLE оказался ровно на эпохе `6`.

Это boundary optimum -- архитектуре просто не дали показать, закончила ли она обучение.

Теперь:

```text
MAX_CV_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 8
EARLY_STOPPING_MIN_DELTA = 1e-4
MIN_CV_EPOCHS_BEFORE_STOP = 8
```

Каждый temporal fold обучается независимо до:

- `30` эпох максимум;
- либо early stopping после `8` эпох без нового local best.

Важный момент -- scheduler в CV и final теперь имеет один и тот же `T_max=MAX_CV_EPOCHS`.

Если CV выберет, например, `BEST_EPOCH=12`, final model тоже проходит первые 12 эпох **той же LR-траектории**, а не новый cosine schedule длиной 12 эпох.

Иначе получался бы train/final mismatch.

Global `BEST_EPOCH` выбирается только среди эпох, которые успели пройти **все 3 temporal fold**. Поздняя эпоха одного fold не может победить просто потому, что два других fold уже остановились.

Эта версия сохраняется отдельно в:

```text
models/lstm_hurdle_v4_expanded_es/
```

Старый public-best `lstm_hurdle_v4_robust` не перезаписывается.


## Final ensemble

После выбора `BEST_EPOCH` обучаются четыре независимые копии одной архитектуры:

```text
seed 42
seed 143
seed 67
seed 2026
```

Все четыре обучаются на всех labeled cutoff ровно `BEST_EPOCH` эпох.

Финальный prediction:

```text
pred_log = mean(
    pred_log_seed42,
    pred_log_seed143,
    pred_log_seed67,
    pred_log_seed2026
)
```

Усреднение делается в `log1p`-пространстве, потом `expm1`.

Сиды не участвуют в выборе `BEST_EPOCH` -- число эпох выбирается temporal CV один раз для архитектуры.


In [1]:
from pathlib import Path
import hashlib
import json
import math
import random
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import ConcatDataset, DataLoader, Dataset


def find_project_root():
    for path in [Path.cwd(), *Path.cwd().parents]:
        if (path / "data" / "lstm" / "meta.json").exists():
            return path
    raise FileNotFoundError(
        "Не найден data/lstm/meta.json. Сначала запустите обновленный 06_LSTM_Data_Preparation.ipynb."
    )


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "lstm"
MODEL_DIR = PROJECT_ROOT / "models" / "lstm_hurdle_v4_expanded_es"
REFERENCE_MODEL_DIR = PROJECT_ROOT / "models" / "lstm_hurdle_v4_robust"
SUBMISSION_DIR = PROJECT_ROOT / "submissions"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

with open(DATA_DIR / "meta.json", encoding="utf-8") as f:
    META = json.load(f)

if META.get("format_version") != 2:
    raise RuntimeError(
        "LSTM v4 ожидает data format v2. Перезапустите обновленный 06_LSTM_Data_Preparation.ipynb."
    )

LABELED_CUTOFFS = META["labeled_cutoffs"]
INFERENCE_CUTOFF = META["inference_cutoff"]
BASE_SEQUENCE_FEATURES = META["base_sequence_features"]
CALENDAR_FEATURES = META["calendar_sequence_features"]
RAW_STATIC_FEATURES = META["static_features"]
RAW_STATIC_LOG_FEATURES = META["static_log_copy_features"]
EXTRA_STATIC_FEATURES = set(META.get("extra_static_features", []))
SEQ_LEN = int(META["seq_len"])
N_USERS = int(META["user_count"])

BASE_INDEX = {name: i for i, name in enumerate(BASE_SEQUENCE_FEATURES)}

# Флаги меняю по одному после baseline run.
USE_USER_EMBEDDING = False
USER_EMBED_DIM = 4
USE_CONV_STEM = False

USE_ABSOLUTE_TIME = True
USE_FUTURE_CALENDAR_STATIC = False
USE_CUTOFF_SEASONAL_STATIC = False

BATCH_SIZE = 768
MAX_CV_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 8
EARLY_STOPPING_MIN_DELTA = 1e-4
MIN_CV_EPOCHS_BEFORE_STOP = 8
N_CV_FOLDS = 3
LR = 3.5e-4
EMBED_LR = 1.0e-4
WEIGHT_DECAY = 1.2e-3
EMBED_WEIGHT_DECAY = 8e-3
RANDOM_STATE = 42
FINAL_SEEDS = [42, 143, 2026]

AUX_GATE_WEIGHT = 0.03
AUX_POS_WEIGHT = 0.10
AUX_DIRECT_WEIGHT = 0.05

REFERENCE_MEAN_CV_RMSLE = 1.715288
REFERENCE_JAN_HOLDOUT_RMSLE = 1.675888
AUTO_SKIP_FINAL_IF_WEAK = True
MAX_MEAN_CV_DEGRADATION_FOR_FINAL = 0.003
MAX_JAN_DEGRADATION_FOR_FINAL = 0.006
FORCE_FINAL_TRAIN = False

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
NUM_WORKERS = 4 if DEVICE.type == "cuda" else 0
USE_AMP = DEVICE.type == "cuda"


def set_seed(seed=RANDOM_STATE):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def choose_static_features():
    selected = []
    for name in RAW_STATIC_FEATURES:
        if name not in EXTRA_STATIC_FEATURES:
            selected.append(name)
            continue

        if name == "cutoff_time_years" and USE_ABSOLUTE_TIME:
            selected.append(name)
        elif name.startswith("forecast_") and USE_FUTURE_CALENDAR_STATIC:
            selected.append(name)
        elif (
            name.startswith("cutoff_")
            and name != "cutoff_time_years"
            and USE_CUTOFF_SEASONAL_STATIC
        ):
            selected.append(name)
    return selected


STATIC_FEATURES = choose_static_features()
STATIC_INDICES = np.array(
    [RAW_STATIC_FEATURES.index(name) for name in STATIC_FEATURES],
    dtype=np.int64,
)
STATIC_LOG_FEATURES = [name for name in RAW_STATIC_LOG_FEATURES if name in STATIC_FEATURES]
STATIC_LOG_INDICES = [STATIC_FEATURES.index(name) for name in STATIC_LOG_FEATURES]

set_seed()
print("device:", DEVICE)
print("AMP:", USE_AMP)
print("labeled cutoffs:", LABELED_CUTOFFS)
print("CV validation cutoffs:", LABELED_CUTOFFS[-N_CV_FOLDS:])
print("inference:", INFERENCE_CUTOFF)
print("users:", N_USERS)
print("static features:", len(STATIC_FEATURES), "/", len(RAW_STATIC_FEATURES))
print("user embedding:", USE_USER_EMBEDDING)
print("future calendar static:", USE_FUTURE_CALENDAR_STATIC)
print("max CV epochs:", MAX_CV_EPOCHS)
print("early stopping patience:", EARLY_STOPPING_PATIENCE)
print("early stopping min delta:", EARLY_STOPPING_MIN_DELTA)
print("minimum epochs before stop:", MIN_CV_EPOCHS_BEFORE_STOP)
print("model dir:", MODEL_DIR)

device: mps
AMP: False
labeled cutoffs: ['2025-04-19', '2025-05-19', '2025-06-18', '2025-07-18', '2025-08-17', '2025-09-16', '2025-10-16', '2025-11-15', '2025-12-15', '2026-01-14']
CV validation cutoffs: ['2025-11-15', '2025-12-15', '2026-01-14']
inference: 2026-02-13
users: 250000
static features: 92 / 103
user embedding: False
future calendar static: False
max CV epochs: 30
early stopping patience: 8
early stopping min delta: 0.0001
minimum epochs before stop: 8
model dir: /Users/pinta/Dev/E-CUP-2026/models/lstm_hurdle_v4_expanded_es


## Dataset и variable-length history

`HybridDataset` возвращает:

- 90-дневную sequence;
- static-признаки;
- `user_index`;
- `user_known`;
- `history_length`;
- target для train cutoff.

`history_length` отделяет реальную историю от pre-first-seen padding.

Если `user_id` embedding включен, unseen user на validation/inference получает нулевой embedding. В финальном run embedding выключен.

In [2]:
def known_user_mask_from_cutoffs(cutoffs):
    mask = np.zeros(N_USERS, dtype=np.bool_)
    for cutoff in cutoffs:
        idx = np.load(DATA_DIR / cutoff / "user_index.npy", mmap_mode="r")
        mask[np.asarray(idx, dtype=np.int64)] = True
    return mask


class HybridDataset(Dataset):
    def __init__(self, cutoff, with_target=True, known_user_mask=None):
        path = DATA_DIR / cutoff
        self.X = np.load(path / "X.npy", mmap_mode="r")
        self.static = np.load(path / "static.npy", mmap_mode="r")
        self.calendar = np.load(path / "calendar.npy", mmap_mode="r").astype(np.float32)
        self.users = np.load(path / "user_id.npy", mmap_mode="r")
        self.user_index = np.load(path / "user_index.npy", mmap_mode="r")
        self.history_length = np.load(path / "history_length.npy", mmap_mode="r")
        self.y = np.load(path / "y.npy", mmap_mode="r") if with_target else None
        self.known_user_mask = known_user_mask

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        sequence = np.concatenate([
            np.asarray(self.X[i], dtype=np.float32),
            self.calendar,
        ], axis=1)

        user_index = int(self.user_index[i])
        user_known = True if self.known_user_mask is None else bool(self.known_user_mask[user_index])

        raw_static = np.asarray(self.static[i], dtype=np.float32)
        static = raw_static[STATIC_INDICES]

        result = (
            torch.from_numpy(sequence),
            torch.from_numpy(static),
            torch.tensor(user_index, dtype=torch.long),
            torch.tensor(user_known, dtype=torch.bool),
            torch.tensor(int(self.history_length[i]), dtype=torch.long),
        )

        if self.y is None:
            return result

        return (*result, torch.tensor(float(self.y[i]), dtype=torch.float32))


def make_loader(
    cutoffs,
    shuffle=False,
    with_target=True,
    batch_size=BATCH_SIZE,
    known_user_mask=None,
):
    if isinstance(cutoffs, str):
        cutoffs = [cutoffs]

    dataset = ConcatDataset([
        HybridDataset(
            cutoff,
            with_target=with_target,
            known_user_mask=known_user_mask,
        )
        for cutoff in cutoffs
    ])

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == "cuda",
        persistent_workers=NUM_WORKERS > 0,
        drop_last=False,
    )

## Признаки

### Sequence -- 53 признака на каждый из 90 дней

19 признаков приходят с диска:

- 13 базовых дневных -- `search`, `cat`, `searches`, переходы, `to_cart`, `to_ord`, GMV и `active`;
- 6 calendar -- sin/cos для дня недели, дня месяца и дня года.

Еще 34 считаю на батче:

- `valid_mask` -- реальная история, без pre-first-seen padding;
- `relative_history_position` -- позиция дня внутри реальной истории;
- `days_to_cutoff` -- нормированное расстояние до cutoff;
- 4 `has_*` -- были ли `search_to_cart`, `search_to_ord`, `cat_to_cart`, `cat_to_ord`;
- 4 ratio -- `search_to_cart/searches`, `search_to_ord/searches`, `to_ord/to_cart`, `gmv_search/gmv`;
- rolling mean 3/7/14/30 дней для `searches`, `to_cart`, `to_ord`, `gmv` -- 16 признаков;
- rolling activity rate 3/7/14/30 -- 4 признака;
- first difference для `searches`, `to_ord`, `gmv` -- 3 признака.

Все rolling и difference считают только валидную историю до cutoff.

### Short summary -- 93 признака на пользователя

Отдельно от BiLSTM считаю summary по последним окнам:

- `log1p(sum)` за 1/3/7/14/30/60/90 дней для `searches`, `to_cart`, `to_ord`, `gmv`, `gmv_search` -- 35;
- activity rate на тех же окнах -- 7;
- days since last event для этих 5 сигналов -- 5;
- recent minus previous за 3/7/14/30 дней для этих 5 сигналов -- 20;
- три непересекающихся 30-дневных bucket + разница recent vs old для этих 5 сигналов -- 20;
- purchase periodicity -- event rate, days since last order, last order gap, phase, наличие двух покупок -- 5;
- `history_length / 90` -- 1.

### Static -- 235 признаков после preprocessing

В финальном run беру:

- 91 исходный признак из `Prepared_data.parquet`;
- `cutoff_time_years` -- один smooth absolute-time признак;
- 51 `log1p`-копию heavy-tail static-признаков;
- 92 missing-mask.

Future-horizon calendar и cutoff-seasonal static по умолчанию выключены.

Причина -- таких значений всего 10 по train cutoff, поэтому богатый calendar легко начинает работать как cutoff ID.

In [3]:
def history_mask_from_lengths(lengths, steps=SEQ_LEN):
    positions = torch.arange(steps, device=lengths.device).unsqueeze(0)
    starts = steps - lengths.unsqueeze(1)
    return positions >= starts


def causal_masked_mean(values, valid_mask, window):
    values = values * valid_mask
    num = F.avg_pool1d(
        F.pad(values.unsqueeze(1), (window - 1, 0)),
        window,
        stride=1,
    ).squeeze(1) * window
    den = F.avg_pool1d(
        F.pad(valid_mask.unsqueeze(1), (window - 1, 0)),
        window,
        stride=1,
    ).squeeze(1) * window
    return num / den.clamp_min(1.0)


def first_difference(values, valid_mask):
    previous_valid = F.pad(valid_mask[:, :-1], (1, 0))
    both_valid = valid_mask * previous_valid
    diff = F.pad(values[:, 1:] - values[:, :-1], (1, 0))
    return diff * both_valid


def safe_ratio(numerator, denominator, max_value=5.0):
    ratio = numerator / denominator.clamp_min(1e-3)
    ratio = torch.where(denominator > 0, ratio, torch.zeros_like(ratio))
    return ratio.clamp(0, max_value)


def make_sequence_features(x, lengths):
    steps = x.shape[1]
    valid_mask = history_mask_from_lengths(lengths, steps).float()

    searches_log = x[..., BASE_INDEX["searches"]]
    search_to_cart_log = x[..., BASE_INDEX["search_to_cart"]]
    search_to_ord_log = x[..., BASE_INDEX["search_to_ord"]]
    cat_to_cart_log = x[..., BASE_INDEX["cat_to_cart"]]
    cat_to_ord_log = x[..., BASE_INDEX["cat_to_ord"]]
    to_cart_log = x[..., BASE_INDEX["to_cart"]]
    to_ord_log = x[..., BASE_INDEX["to_ord"]]
    gmv_search_log = x[..., BASE_INDEX["gmv_search"]]
    gmv_log = x[..., BASE_INDEX["gmv"]]
    active = x[..., BASE_INDEX["active"]]

    searches = torch.expm1(searches_log).clamp_min(0)
    search_to_cart = torch.expm1(search_to_cart_log).clamp_min(0)
    search_to_ord = torch.expm1(search_to_ord_log).clamp_min(0)
    cat_to_cart = torch.expm1(cat_to_cart_log).clamp_min(0)
    cat_to_ord = torch.expm1(cat_to_ord_log).clamp_min(0)
    to_cart = torch.expm1(to_cart_log).clamp_min(0)
    to_ord = torch.expm1(to_ord_log).clamp_min(0)
    gmv_search = torch.expm1(gmv_search_log).clamp_min(0)
    gmv = torch.expm1(gmv_log).clamp_min(0)

    positions = torch.arange(steps, device=x.device).float().unsqueeze(0)
    starts = (steps - lengths).float().unsqueeze(1)
    denom = (lengths.float() - 1).clamp_min(1.0).unsqueeze(1)

    relative_history_position = ((positions - starts) / denom).clamp(0, 1) * valid_mask
    days_to_cutoff = ((steps - 1 - positions) / max(steps - 1, 1)) * valid_mask

    derived = [
        valid_mask,
        relative_history_position,
        days_to_cutoff,
        (search_to_cart > 0).float(),
        (search_to_ord > 0).float(),
        (cat_to_cart > 0).float(),
        (cat_to_ord > 0).float(),
        safe_ratio(search_to_cart, searches),
        safe_ratio(search_to_ord, searches),
        safe_ratio(to_ord, to_cart),
        safe_ratio(gmv_search, gmv, 1.5),
    ]

    for values in [searches_log, to_cart_log, to_ord_log, gmv_log]:
        for window in [3, 7, 14, 30]:
            derived.append(causal_masked_mean(values, valid_mask, window))

    for window in [3, 7, 14, 30]:
        derived.append(causal_masked_mean(active, valid_mask, window))

    for values in [searches_log, to_ord_log, gmv_log]:
        derived.append(first_difference(values, valid_mask))

    return torch.cat([x, torch.stack(derived, dim=-1)], dim=-1)


def _raw_channel(x, feature_name):
    return torch.expm1(x[..., BASE_INDEX[feature_name]]).clamp_min(0)


def _window_raw_sum(x, feature_name, window):
    return _raw_channel(x, feature_name)[:, -window:].sum(dim=1)


def _slice_raw_sum(x, feature_name, start_from_end, end_from_end):
    values = _raw_channel(x, feature_name)
    left = x.shape[1] - start_from_end
    right = x.shape[1] - end_from_end if end_from_end > 0 else x.shape[1]
    return values[:, max(left, 0):max(right, 0)].sum(dim=1)


def _days_since_event(x, lengths, feature_name):
    values = _raw_channel(x, feature_name)
    valid = history_mask_from_lengths(lengths, x.shape[1])
    event = (values > 0) & valid
    positions = torch.arange(x.shape[1], device=x.device).unsqueeze(0).expand_as(values)
    last = torch.where(event, positions, torch.full_like(positions, -1)).amax(dim=1)
    days = (x.shape[1] - 1 - last).float()
    return torch.where(last >= 0, days / x.shape[1], torch.full_like(days, 1.25))


def _purchase_periodicity(x, lengths):
    values = _raw_channel(x, "to_ord")
    valid = history_mask_from_lengths(lengths, x.shape[1])
    event = (values > 0) & valid

    positions = torch.arange(x.shape[1], device=x.device).unsqueeze(0).expand_as(values)
    masked_pos = torch.where(event, positions, torch.full_like(positions, -1))
    top2 = torch.topk(masked_pos, k=2, dim=1).values
    last = top2[:, 0]
    previous = top2[:, 1]

    has_any = last >= 0
    has_two = previous >= 0

    days_since = torch.where(
        has_any,
        (x.shape[1] - 1 - last).float() / x.shape[1],
        torch.full_like(last.float(), 1.25),
    )
    last_gap = torch.where(
        has_two,
        (last - previous).float() / x.shape[1],
        torch.ones_like(last.float()),
    )
    event_rate = event.sum(dim=1).float() / lengths.float().clamp_min(1.0)
    phase = torch.where(
        has_two,
        days_since / last_gap.clamp_min(1.0 / x.shape[1]),
        torch.ones_like(days_since),
    ).clamp(0, 5)

    return [event_rate, days_since, last_gap, phase, has_two.float()]


def make_short_summary(x, lengths):
    windows = [1, 3, 7, 14, 30, 60, 90]
    volume_names = ["searches", "to_cart", "to_ord", "gmv", "gmv_search"]
    features = []

    for name in volume_names:
        for window in windows:
            features.append(torch.log1p(_window_raw_sum(x, name, window)))

    active = x[..., BASE_INDEX["active"]]
    for window in windows:
        exposure = lengths.clamp(max=window).float()
        features.append(active[:, -window:].sum(dim=1) / exposure.clamp_min(1.0))

    for name in volume_names:
        features.append(_days_since_event(x, lengths, name))

    for name in volume_names:
        values = _raw_channel(x, name)
        for window in [3, 7, 14, 30]:
            recent = values[:, -window:].sum(dim=1)
            previous = values[:, -2 * window:-window].sum(dim=1)
            features.append(torch.log1p(recent) - torch.log1p(previous))

    for name in volume_names:
        buckets = [
            _slice_raw_sum(x, name, 30, 0),
            _slice_raw_sum(x, name, 60, 30),
            _slice_raw_sum(x, name, 90, 60),
        ]
        bucket_logs = [torch.log1p(v) for v in buckets]
        features.extend(bucket_logs)
        features.append(bucket_logs[0] - bucket_logs[2])

    features.extend(_purchase_periodicity(x, lengths))
    features.append(lengths.float() / x.shape[1])

    return torch.stack(features, dim=1)


with torch.no_grad():
    dummy_x = torch.zeros(
        2,
        SEQ_LEN,
        len(BASE_SEQUENCE_FEATURES) + len(CALENDAR_FEATURES),
        dtype=torch.float32,
    )
    dummy_lengths = torch.tensor([SEQ_LEN, max(2, SEQ_LEN // 2)], dtype=torch.long)
    SEQ_INPUT_SIZE = int(make_sequence_features(dummy_x, dummy_lengths).shape[-1])
    SHORT_SUMMARY_SIZE = int(make_short_summary(dummy_x, dummy_lengths).shape[-1])

print("sequence features:", SEQ_INPUT_SIZE)
print("short summary:", SHORT_SUMMARY_SIZE)

sequence features: 53
short summary: 93


## Static preprocessing

На каждом CV fold preprocessing фитится только по train cutoff.

Схема:

- выбираю 92 raw static-признака;
- добавляю 51 `log1p`-копию heavy-tail признаков;
- добавляю 92 missing-mask;
- пропуски заменяю train mean;
- стандартизую train mean/std.

Validation в расчет mean/std не попадает.

In [4]:
STATIC_INPUT_SIZE = 2 * len(STATIC_FEATURES) + len(STATIC_LOG_INDICES)


def augment_static_numpy(raw):
    raw = np.asarray(raw, dtype=np.float32)
    source = raw[:, STATIC_LOG_INDICES]
    logs = np.where(
        np.isfinite(source),
        np.log1p(np.clip(source, 0, None)),
        np.nan,
    ).astype(np.float32)
    missing = (~np.isfinite(raw)).astype(np.float32)
    return np.concatenate([raw, logs, missing], axis=1)


def fit_static_stats(cutoffs, chunk_size=65_536):
    sums = np.zeros(STATIC_INPUT_SIZE, dtype=np.float64)
    sums_sq = np.zeros(STATIC_INPUT_SIZE, dtype=np.float64)
    counts = np.zeros(STATIC_INPUT_SIZE, dtype=np.int64)

    for cutoff in cutoffs:
        raw_full = np.load(DATA_DIR / cutoff / "static.npy", mmap_mode="r")
        for start in range(0, len(raw_full), chunk_size):
            raw = np.asarray(raw_full[start:start + chunk_size], dtype=np.float32)[:, STATIC_INDICES]
            block = augment_static_numpy(raw)
            finite = np.isfinite(block)
            safe = np.where(finite, block, 0.0).astype(np.float64)
            sums += safe.sum(axis=0)
            sums_sq += (safe * safe).sum(axis=0)
            counts += finite.sum(axis=0)

    counts = np.maximum(counts, 1)
    mean = sums / counts
    std = np.sqrt(np.maximum(sums_sq / counts - mean * mean, 1e-6))
    return (
        torch.tensor(mean, dtype=torch.float32),
        torch.tensor(std, dtype=torch.float32),
    )


def normalize_static(raw, stats):
    source = raw[:, STATIC_LOG_INDICES]
    logs = torch.where(
        torch.isfinite(source),
        torch.log1p(source.clamp_min(0)),
        torch.nan,
    )
    augmented = torch.cat([raw, logs, (~torch.isfinite(raw)).float()], dim=1)

    mean, std = stats
    mean = mean.to(raw.device)
    std = std.to(raw.device)

    augmented = torch.where(torch.isfinite(augmented), augmented, mean)
    return (augmented - mean) / std


print("static features after augmentation:", STATIC_INPUT_SIZE)

static features after augmentation: 235


## Текущая архитектура

Sequence-ветка:

`53/day -> Linear(80) -> LayerNorm -> GELU -> 2-layer BiLSTM(hidden=112)`

После BiLSTM беру четыре pooling:

- last hidden;
- masked mean;
- masked max;
- attention pooling с обучаемым recency bias.

Их concat идет в `sequence_head -> 144`.

Отдельные ветки:

- `short summary: 93 -> 96 -> 64`;
- `static: 235 -> 192 -> 96`;
- optional `user_id` embedding -- в финальном run выключен.

После concat:

`144 + 64 + 96 -> fusion -> 96`

Из общего representation выходят три головы:

- `gate_head` -- `P(GMV > 0)`;
- `positive_head` -- `log1p(GMV)` для положительной части;
- `direct_head` -- прямой прогноз `E[log1p(GMV) | x]`.

Hurdle-прогноз:

$$
\hat z_{hurdle}=P(GMV>0)\cdot \hat z_{positive}.
$$

Финальный прогноз:

$$
\hat z=w\hat z_{hurdle}+(1-w)\hat z_{direct},
$$

где `w` -- обучаемый scalar.

Conv stem и `user_id` embedding оставлены в коде только как отдельные ablation. В финальном run оба выключены.

In [5]:
def compact_left_padded(sequence, lengths):
    # Left padding переношу вправо перед packing.
    batch, steps, channels = sequence.shape
    positions = torch.arange(steps, device=sequence.device).unsqueeze(0).expand(batch, -1)
    starts = steps - lengths.unsqueeze(1)
    source_pos = (positions + starts).clamp(max=steps - 1)
    gathered = sequence.gather(
        1,
        source_pos.unsqueeze(-1).expand(-1, -1, channels),
    )
    valid = positions < lengths.unsqueeze(1)
    return gathered * valid.unsqueeze(-1)


class ResidualConvBlock(nn.Module):
    def __init__(self, channels, kernel_size=5, dropout=0.10):
        super().__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(channels, channels, kernel_size, padding=padding)
        self.conv2 = nn.Conv1d(channels, channels, 3, padding=1)
        self.norm1 = nn.LayerNorm(channels)
        self.norm2 = nn.LayerNorm(channels)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, valid_mask):
        residual = x
        y = self.conv1(x.transpose(1, 2)).transpose(1, 2)
        y = self.norm1(y)
        y = F.gelu(y)
        y = self.dropout(y)
        y = self.conv2(y.transpose(1, 2)).transpose(1, 2)
        y = self.norm2(y)
        y = F.gelu(y)
        y = self.dropout(y)
        return (residual + y) * valid_mask.unsqueeze(-1)


class JointHurdleLSTM(nn.Module):
    def __init__(
        self,
        n_users,
        user_embed_dim=0,
        d_model=80,
        hidden_size=112,
        num_layers=2,
        lstm_dropout=0.25,
        head_dropout=0.35,
        entity_dropout=0.60,
        use_conv_stem=False,
    ):
        super().__init__()

        self.hidden_size = hidden_size
        self.user_embed_dim = int(user_embed_dim)
        self.entity_dropout = float(entity_dropout)
        self.use_conv_stem = bool(use_conv_stem)

        self.sequence_projection = nn.Sequential(
            nn.Linear(SEQ_INPUT_SIZE, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(0.10),
        )

        self.conv_stem = (
            ResidualConvBlock(d_model, kernel_size=5, dropout=0.10)
            if self.use_conv_stem
            else None
        )

        self.lstm = nn.LSTM(
            input_size=d_model,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=lstm_dropout if num_layers > 1 else 0.0,
            bidirectional=True,
        )

        self.attention_score = nn.Sequential(
            nn.Linear(hidden_size * 2, 48),
            nn.Tanh(),
            nn.Linear(48, 1),
        )
        self.recency_attention_bias = nn.Parameter(torch.tensor(0.20))

        pooled_size = hidden_size * 2 * 4
        self.sequence_head = nn.Sequential(
            nn.LayerNorm(pooled_size),
            nn.Linear(pooled_size, 192),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(192, 144),
            nn.GELU(),
        )

        self.summary_head = nn.Sequential(
            nn.LayerNorm(SHORT_SUMMARY_SIZE),
            nn.Linear(SHORT_SUMMARY_SIZE, 96),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(96, 64),
            nn.GELU(),
        )

        self.static_head = nn.Sequential(
            nn.Linear(STATIC_INPUT_SIZE, 192),
            nn.LayerNorm(192),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(192, 96),
            nn.GELU(),
            nn.Dropout(0.20),
        )

        if self.user_embed_dim > 0:
            self.user_embedding = nn.Embedding(n_users, self.user_embed_dim, max_norm=1.0)
            nn.init.normal_(self.user_embedding.weight, mean=0.0, std=0.015)
            self.user_scale_logit = nn.Parameter(torch.tensor(-3.0))
        else:
            self.user_embedding = None
            self.register_parameter("user_scale_logit", None)

        fusion_size = 144 + 64 + 96 + self.user_embed_dim
        self.fusion = nn.Sequential(
            nn.Linear(fusion_size, 192),
            nn.LayerNorm(192),
            nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(192, 96),
            nn.GELU(),
            nn.Dropout(0.20),
        )

        self.gate_head = nn.Linear(96, 1)
        self.positive_head = nn.Linear(96, 1)
        self.direct_head = nn.Linear(96, 1)
        self.hurdle_mix_logit = nn.Parameter(torch.tensor(math.log(0.6 / 0.4)))

    def _user_repr(self, user_index, user_known):
        if self.user_embedding is None:
            return torch.empty(len(user_index), 0, device=user_index.device)

        emb = self.user_embedding(user_index)
        emb = emb * user_known.float().unsqueeze(1)

        if self.training and self.entity_dropout > 0:
            keep = (
                torch.rand((len(emb), 1), device=emb.device) >= self.entity_dropout
            ).float()
            emb = emb * keep / (1.0 - self.entity_dropout)

        scale = torch.sigmoid(self.user_scale_logit)
        return emb * scale

    def forward(self, sequence, static, user_index, user_known, history_length):
        short_summary = make_short_summary(sequence, history_length)

        sequence = make_sequence_features(sequence, history_length)
        sequence = compact_left_padded(sequence, history_length)

        positions = torch.arange(SEQ_LEN, device=sequence.device).unsqueeze(0)
        valid = positions < history_length.unsqueeze(1)
        valid_f = valid.float()

        sequence = self.sequence_projection(sequence) * valid_f.unsqueeze(-1)

        if self.conv_stem is not None:
            sequence = self.conv_stem(sequence, valid_f)

        packed = pack_padded_sequence(
            sequence,
            history_length.detach().cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        packed_output, (hidden, _) = self.lstm(packed)
        output, _ = pad_packed_sequence(
            packed_output,
            batch_first=True,
            total_length=SEQ_LEN,
        )

        valid3 = valid.unsqueeze(-1)
        pooled_mean = (output * valid3).sum(dim=1) / history_length.float().unsqueeze(1)
        pooled_max = output.masked_fill(~valid3, float("-inf")).amax(dim=1)
        last_hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)

        scores = self.attention_score(output).squeeze(-1)
        relative_pos = positions.float() / (
            history_length.unsqueeze(1).float() - 1
        ).clamp_min(1.0)
        scores = scores + self.recency_attention_bias * relative_pos
        scores = scores.masked_fill(~valid, -1e9)
        attn_weight = torch.softmax(scores, dim=1)
        pooled_attn = (output * attn_weight.unsqueeze(-1)).sum(dim=1)

        sequence_repr = self.sequence_head(torch.cat([
            last_hidden,
            pooled_mean,
            pooled_max,
            pooled_attn,
        ], dim=1))
        summary_repr = self.summary_head(short_summary)
        static_repr = self.static_head(static)
        user_repr = self._user_repr(user_index, user_known)

        fused = self.fusion(torch.cat([
            sequence_repr,
            summary_repr,
            static_repr,
            user_repr,
        ], dim=1))

        gate_logit = self.gate_head(fused).squeeze(1)
        positive_log = F.softplus(self.positive_head(fused).squeeze(1))
        direct_log = F.softplus(self.direct_head(fused).squeeze(1))

        gate_prob = torch.sigmoid(gate_logit)
        hurdle_log = gate_prob * positive_log
        hurdle_weight = torch.sigmoid(self.hurdle_mix_logit)

        pred_log = hurdle_weight * hurdle_log + (1.0 - hurdle_weight) * direct_log

        user_scale = (
            torch.sigmoid(self.user_scale_logit)
            if self.user_scale_logit is not None
            else torch.tensor(0.0, device=pred_log.device)
        )

        return {
            "pred_log": pred_log,
            "gate_logit": gate_logit,
            "gate_prob": gate_prob,
            "positive_log": positive_log,
            "direct_log": direct_log,
            "hurdle_log": hurdle_log,
            "hurdle_weight": hurdle_weight,
            "user_scale": user_scale,
        }


MODEL_KWARGS = {
    "n_users": N_USERS,
    "user_embed_dim": USER_EMBED_DIM if USE_USER_EMBEDDING else 0,
    "d_model": 80,
    "hidden_size": 112,
    "num_layers": 2,
    "lstm_dropout": 0.25,
    "head_dropout": 0.35,
    "entity_dropout": 0.60,
    "use_conv_stem": USE_CONV_STEM,
}

probe = JointHurdleLSTM(**MODEL_KWARGS)
print("parameters:", f"{sum(p.numel() for p in probe.parameters()):,}")
del probe

parameters: 851,216


## Loss и optimizer

Работаю сразу в `log1p`-пространстве:

$$
z=\log(1+y).
$$

Главный loss:

`MSE(pred_log, z)`

Это MSLE, то есть RMSLE без квадратного корня. Точка минимума та же.

Дополнительные loss:

- `0.03 * BCE(gate, y > 0)`;
- `0.10 * MSE(positive_log, z)` только для `y > 0`;
- `0.05 * MSE(direct_log, z)`.

Главный objective всегда считает финальный blended prediction.

Optimizer -- `AdamW`, gradient clipping `1.0`, cosine LR schedule. AMP включается только на CUDA.

In [6]:
def unpack_batch(batch, with_target=True):
    if with_target:
        sequence, static, user_index, user_known, history_length, y = batch
        y = y.to(DEVICE, non_blocking=True)
    else:
        sequence, static, user_index, user_known, history_length = batch
        y = None

    return (
        sequence.to(DEVICE, non_blocking=True),
        static.to(DEVICE, non_blocking=True),
        user_index.to(DEVICE, non_blocking=True),
        user_known.to(DEVICE, non_blocking=True),
        history_length.to(DEVICE, non_blocking=True),
        y,
    )


def compute_joint_loss(out, y):
    target_log = torch.log1p(y)
    nonzero = (y > 0).float()

    main = F.mse_loss(out["pred_log"], target_log)
    gate = F.binary_cross_entropy_with_logits(out["gate_logit"], nonzero)

    positive_mask = y > 0
    if positive_mask.any():
        positive = F.mse_loss(
            out["positive_log"][positive_mask],
            target_log[positive_mask],
        )
    else:
        positive = out["positive_log"].sum() * 0.0

    direct = F.mse_loss(out["direct_log"], target_log)

    total = (
        main
        + AUX_GATE_WEIGHT * gate
        + AUX_POS_WEIGHT * positive
        + AUX_DIRECT_WEIGHT * direct
    )
    return total, {
        "main_mse": main.detach(),
        "gate_bce": gate.detach(),
        "positive_mse": positive.detach(),
        "direct_mse": direct.detach(),
    }


def make_optimizer(model):
    if model.user_embedding is None:
        return torch.optim.AdamW(
            model.parameters(),
            lr=LR,
            weight_decay=WEIGHT_DECAY,
        )

    embed_ids = {id(p) for p in model.user_embedding.parameters()}
    embed_params = [p for p in model.parameters() if id(p) in embed_ids]
    main_params = [p for p in model.parameters() if id(p) not in embed_ids]

    return torch.optim.AdamW([
        {
            "params": main_params,
            "lr": LR,
            "weight_decay": WEIGHT_DECAY,
        },
        {
            "params": embed_params,
            "lr": EMBED_LR,
            "weight_decay": EMBED_WEIGHT_DECAY,
        },
    ])


def new_grad_scaler():
    if not USE_AMP:
        return None
    try:
        return torch.amp.GradScaler("cuda", enabled=True)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=True)


def train_one_epoch(model, loader, optimizer, static_stats, scaler=None):
    model.train()
    totals = {}
    total_n = 0

    for batch in loader:
        sequence, static, user_index, user_known, history_length, y = unpack_batch(batch)
        static = normalize_static(static, static_stats)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=USE_AMP,
        ):
            out = model(sequence, static, user_index, user_known, history_length)
            loss, parts = compute_joint_loss(out, y)

        if scaler is not None and USE_AMP:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        n = len(y)
        totals["loss"] = totals.get("loss", 0.0) + float(loss.detach().cpu()) * n
        for key, value in parts.items():
            totals[key] = totals.get(key, 0.0) + float(value.cpu()) * n
        total_n += n

    return {k: v / max(total_n, 1) for k, v in totals.items()}


@torch.no_grad()
def predict_model(model, cutoff, static_stats, known_user_mask, with_target=True):
    dataset = HybridDataset(
        cutoff,
        with_target=with_target,
        known_user_mask=known_user_mask,
    )
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == "cuda",
        persistent_workers=NUM_WORKERS > 0,
    )

    model.eval()
    collected = {
        "pred_log": [],
        "gate_logit": [],
        "gate_prob": [],
        "positive_log": [],
        "direct_log": [],
        "hurdle_log": [],
    }
    targets = []

    for batch in loader:
        sequence, static, user_index, user_known, history_length, y = unpack_batch(
            batch,
            with_target=with_target,
        )
        static = normalize_static(static, static_stats)
        out = model(sequence, static, user_index, user_known, history_length)

        for key in collected:
            collected[key].append(out[key].detach().float().cpu().numpy())
        if with_target:
            targets.append(y.float().cpu().numpy())

    collected = {
        key: np.concatenate(value).astype(np.float32)
        for key, value in collected.items()
    }
    y = np.concatenate(targets).astype(np.float32) if with_target else None

    return np.asarray(dataset.users), collected, y


def _rmsle_from_log(target_log, pred_log):
    pred_log = np.clip(pred_log, 0, None)
    return float(np.sqrt(np.mean((target_log - pred_log) ** 2)))


def validation_metrics(y, pred, model):
    target_log = np.log1p(y)
    nonzero = (y > 0).astype(np.float32)

    p = np.clip(pred["gate_prob"], 1e-6, 1 - 1e-6)
    gate_bce = float(
        -np.mean(nonzero * np.log(p) + (1 - nonzero) * np.log(1 - p))
    )

    try:
        gate_auc = float(roc_auc_score(nonzero, p))
    except ValueError:
        gate_auc = float("nan")

    positive_mask = y > 0
    positive_rmse = float(np.sqrt(np.mean(
        (target_log[positive_mask] - pred["positive_log"][positive_mask]) ** 2
    )))

    return {
        "rmsle": _rmsle_from_log(target_log, pred["pred_log"]),
        "direct_rmsle": _rmsle_from_log(target_log, pred["direct_log"]),
        "hurdle_rmsle": _rmsle_from_log(target_log, pred["hurdle_log"]),
        "gate_bce": gate_bce,
        "gate_auc": gate_auc,
        "true_nonzero_rate": float(nonzero.mean()),
        "pred_nonzero_rate": float(p.mean()),
        "positive_rmse": positive_rmse,
        "hurdle_weight": float(torch.sigmoid(model.hurdle_mix_logit).detach().cpu()),
        "user_scale": (
            float(torch.sigmoid(model.user_scale_logit).detach().cpu())
            if model.user_scale_logit is not None
            else 0.0
        ),
    }

## 3-fold expanding temporal CV + early stopping

CV остается expanding temporal:

```text
Nov <- Apr..Oct
Dec <- Apr..Nov
Jan <- Apr..Dec
```

Что изменилось:

- максимум `30` эпох вместо `6`;
- local early stopping отдельно внутри каждого fold;
- patience `8`, `min_delta=1e-4`;
- fold state сохраняется после каждой эпохи;
- после падения kernel CV продолжается с последнего checkpoint;
- global `BEST_EPOCH` выбирается только по эпохам, которые прошли все fold;
- January prediction каждого epoch сохраняется отдельно для последующего error analysis.

Early stopping здесь нужен как compute guardrail. Финальный выбор эпохи все равно делается по mean temporal CV.

In [ ]:
CV_VALID_CUTOFFS = LABELED_CUTOFFS[-N_CV_FOLDS:]

CV_STATE_DIR = MODEL_DIR / "cv_state"
CV_STATE_DIR.mkdir(parents=True, exist_ok=True)

CV_RUN_SIGNATURE = hashlib.sha256(
    json.dumps(
        {
            "architecture": "JointHurdleLSTM-v4-expanded-es",
            "model_kwargs": MODEL_KWARGS,
            "max_cv_epochs": MAX_CV_EPOCHS,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "early_stopping_min_delta": EARLY_STOPPING_MIN_DELTA,
            "min_cv_epochs_before_stop": MIN_CV_EPOCHS_BEFORE_STOP,
            "lr": LR,
            "weight_decay": WEIGHT_DECAY,
            "aux_weights": {
                "gate": AUX_GATE_WEIGHT,
                "positive": AUX_POS_WEIGHT,
                "direct": AUX_DIRECT_WEIGHT,
            },
            "flags": {
                "user_embedding": USE_USER_EMBEDDING,
                "conv_stem": USE_CONV_STEM,
                "absolute_time": USE_ABSOLUTE_TIME,
                "future_calendar": USE_FUTURE_CALENDAR_STATIC,
                "cutoff_seasonal": USE_CUTOFF_SEASONAL_STATIC,
            },
        },
        sort_keys=True,
        default=str,
    ).encode("utf-8")
).hexdigest()[:16]


def _load_cv_checkpoint(path):
    try:
        return torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=DEVICE)


def _save_cv_checkpoint(
    path,
    fold_idx,
    valid_cutoff,
    epoch,
    model,
    optimizer,
    scheduler,
    scaler,
    static_stats,
    rows,
    best_rmsle,
    best_epoch,
    epochs_without_improvement,
    stopped,
):
    payload = {
        "signature": CV_RUN_SIGNATURE,
        "fold": int(fold_idx),
        "valid_cutoff": valid_cutoff,
        "epoch": int(epoch),
        "model_kwargs": MODEL_KWARGS,
        "state_dict": {
            key: value.detach().cpu()
            for key, value in model.state_dict().items()
        },
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
        "static_mean": static_stats[0].cpu(),
        "static_std": static_stats[1].cpu(),
        "rows": rows,
        "best_rmsle": float(best_rmsle),
        "best_epoch": int(best_epoch),
        "epochs_without_improvement": int(epochs_without_improvement),
        "stopped": bool(stopped),
    }

    tmp = path.with_suffix(".tmp")
    torch.save(payload, tmp)
    tmp.replace(path)


def _save_january_prediction(path, users, y_true, pred):
    np.savez_compressed(
        path,
        user_id=users,
        y_true=y_true,
        **{
            key: value
            for key, value in pred.items()
        },
    )


cv_rows = []
fold_stop_rows = []

last_fold_prediction_store = {}
last_fold_users = None
last_fold_y = None


for fold_idx, valid_cutoff in enumerate(CV_VALID_CUTOFFS, start=1):
    valid_pos = LABELED_CUTOFFS.index(valid_cutoff)
    train_cutoffs = LABELED_CUTOFFS[:valid_pos]

    fold_dir = CV_STATE_DIR / f"fold_{fold_idx}_{valid_cutoff}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    resume_path = fold_dir / "resume.pt"

    print("\n" + "=" * 100)
    print(f"FOLD {fold_idx}/{N_CV_FOLDS} | valid={valid_cutoff}")
    print("train:", train_cutoffs)

    set_seed(RANDOM_STATE)

    static_stats = fit_static_stats(train_cutoffs)
    known_user_mask = known_user_mask_from_cutoffs(train_cutoffs)

    train_loader = make_loader(
        train_cutoffs,
        shuffle=True,
        known_user_mask=known_user_mask,
    )

    model = JointHurdleLSTM(**MODEL_KWARGS).to(DEVICE)
    optimizer = make_optimizer(model)
    scaler = new_grad_scaler()

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=MAX_CV_EPOCHS,
        eta_min=2e-5,
    )

    fold_rows = []
    best_rmsle = float("inf")
    best_epoch = 0
    epochs_without_improvement = 0
    start_epoch = 1
    fold_already_finished = False

    if resume_path.exists():
        checkpoint = _load_cv_checkpoint(resume_path)

        if checkpoint.get("signature") == CV_RUN_SIGNATURE:
            fold_rows = checkpoint.get("rows", [])
            best_rmsle = float(checkpoint.get("best_rmsle", float("inf")))
            best_epoch = int(checkpoint.get("best_epoch", 0))
            epochs_without_improvement = int(
                checkpoint.get("epochs_without_improvement", 0)
            )

            start_epoch = int(checkpoint["epoch"]) + 1
            fold_already_finished = bool(checkpoint.get("stopped", False))

            if not fold_already_finished and start_epoch <= MAX_CV_EPOCHS:
                model.load_state_dict(
                    checkpoint["state_dict"],
                    strict=True,
                )

                optimizer.load_state_dict(
                    checkpoint["optimizer_state_dict"]
                )

                scheduler.load_state_dict(
                    checkpoint["scheduler_state_dict"]
                )

                if (
                    scaler is not None
                    and checkpoint.get("scaler_state_dict") is not None
                ):
                    scaler.load_state_dict(
                        checkpoint["scaler_state_dict"]
                    )

                static_stats = (
                    checkpoint["static_mean"].cpu(),
                    checkpoint["static_std"].cpu(),
                )

                print(
                    f"RESUME: epoch {checkpoint['epoch']} completed "
                    f"-> start from {start_epoch}"
                )

            elif fold_already_finished:
                print(
                    f"CACHE: fold already stopped at epoch {checkpoint['epoch']} "
                    f"| local best={best_epoch}"
                )

        else:
            print("STALE CV checkpoint ignored:", resume_path)

    if not fold_already_finished:
        for epoch in range(start_epoch, MAX_CV_EPOCHS + 1):
            train_metrics = train_one_epoch(
                model,
                train_loader,
                optimizer,
                static_stats,
                scaler,
            )

            scheduler.step()

            valid_users, valid_pred, y_valid = predict_model(
                model,
                valid_cutoff,
                static_stats,
                known_user_mask,
                with_target=True,
            )

            metrics = validation_metrics(
                y_valid,
                valid_pred,
                model,
            )

            row = {
                "fold": fold_idx,
                "valid_cutoff": valid_cutoff,
                "epoch": epoch,
                "lr": float(optimizer.param_groups[0]["lr"]),
                "train_loss": train_metrics["loss"],
                "train_main_mse": train_metrics["main_mse"],
                "train_rmsle_proxy": float(
                    np.sqrt(max(train_metrics["main_mse"], 0.0))
                ),
                **metrics,
            }

            fold_rows.append(row)

            if valid_cutoff == LABELED_CUTOFFS[-1]:
                prediction_path = fold_dir / f"prediction_epoch_{epoch:02d}.npz"

                _save_january_prediction(
                    prediction_path,
                    valid_users,
                    y_valid,
                    valid_pred,
                )

            improved = (
                metrics["rmsle"]
                < best_rmsle - EARLY_STOPPING_MIN_DELTA
            )

            if improved:
                best_rmsle = float(metrics["rmsle"])
                best_epoch = int(epoch)
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1

            should_stop = (
                epoch >= MIN_CV_EPOCHS_BEFORE_STOP
                and epochs_without_improvement >= EARLY_STOPPING_PATIENCE
            )

            _save_cv_checkpoint(
                resume_path,
                fold_idx,
                valid_cutoff,
                epoch,
                model,
                optimizer,
                scheduler,
                scaler,
                static_stats,
                fold_rows,
                best_rmsle,
                best_epoch,
                epochs_without_improvement,
                stopped=should_stop or epoch == MAX_CV_EPOCHS,
            )

            print(
                f"epoch {epoch:02d}/{MAX_CV_EPOCHS:02d} | "
                f"lr={row['lr']:.2e} | "
                f"train_RMSE={row['train_rmsle_proxy']:.4f} | "
                f"valid_RMSLE={metrics['rmsle']:.6f} | "
                f"direct={metrics['direct_rmsle']:.6f} | "
                f"hurdle={metrics['hurdle_rmsle']:.6f} | "
                f"gate_BCE={metrics['gate_bce']:.4f} | "
                f"AUC={metrics['gate_auc']:.4f} | "
                f"p_mean={metrics['pred_nonzero_rate']:.3f}/"
                f"{metrics['true_nonzero_rate']:.3f} | "
                f"mix={metrics['hurdle_weight']:.3f} | "
                f"no_improve={epochs_without_improvement}/"
                f"{EARLY_STOPPING_PATIENCE}"
            )

            if should_stop:
                print(
                    f"EARLY STOP: fold={fold_idx} "
                    f"| last_epoch={epoch} "
                    f"| local_best_epoch={best_epoch} "
                    f"| local_best_RMSLE={best_rmsle:.6f}"
                )
                break

    cv_rows.extend(fold_rows)

    fold_last_epoch = max(
        row["epoch"]
        for row in fold_rows
    )

    fold_stop_rows.append({
        "fold": fold_idx,
        "valid_cutoff": valid_cutoff,
        "last_epoch": fold_last_epoch,
        "local_best_epoch": best_epoch,
        "local_best_rmsle": best_rmsle,
    })

    if valid_cutoff == LABELED_CUTOFFS[-1]:
        for row in fold_rows:
            epoch = int(row["epoch"])
            prediction_path = fold_dir / f"prediction_epoch_{epoch:02d}.npz"

            if not prediction_path.exists():
                continue

            saved = np.load(prediction_path)

            if last_fold_users is None:
                last_fold_users = saved["user_id"].copy()
                last_fold_y = saved["y_true"].copy()

            last_fold_prediction_store[epoch] = {
                key: saved[key].copy()
                for key in [
                    "pred_log",
                    "gate_prob",
                    "positive_log",
                    "direct_log",
                    "hurdle_log",
                ]
            }

    del model, optimizer, train_loader

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


cv_df = pd.DataFrame(cv_rows)

epoch_summary = (
    cv_df.groupby("epoch", as_index=False)
    .agg(
        fold_count=("fold", "nunique"),
        mean_rmsle=("rmsle", "mean"),
        std_rmsle=("rmsle", "std"),
        worst_rmsle=("rmsle", "max"),
        mean_direct_rmsle=("direct_rmsle", "mean"),
        mean_hurdle_rmsle=("hurdle_rmsle", "mean"),
        mean_gate_bce=("gate_bce", "mean"),
        mean_auc=("gate_auc", "mean"),
        mean_pos_rmse=("positive_rmse", "mean"),
        mean_lr=("lr", "mean"),
    )
)

cv_summary = (
    epoch_summary[
        epoch_summary["fold_count"] == N_CV_FOLDS
    ]
    .sort_values(
        ["mean_rmsle", "worst_rmsle"],
        ignore_index=True,
    )
)

if cv_summary.empty:
    raise RuntimeError(
        "No epoch was completed by all CV folds"
    )

display(pd.DataFrame(fold_stop_rows))
display(cv_summary)

BEST_EPOCH = int(cv_summary.loc[0, "epoch"])
CV_RMSLE = float(cv_summary.loc[0, "mean_rmsle"])
CV_STD = float(cv_summary.loc[0, "std_rmsle"])

LAST_COMMON_EPOCH = int(
    cv_summary["epoch"].max()
)

jan_rows = cv_df[
    (cv_df["valid_cutoff"] == LABELED_CUTOFFS[-1])
    & (cv_df["epoch"] == BEST_EPOCH)
]

if len(jan_rows) != 1:
    raise RuntimeError(
        f"January prediction for BEST_EPOCH={BEST_EPOCH} is unavailable"
    )

JAN_RMSLE = float(
    jan_rows.iloc[0]["rmsle"]
)

print("BEST_EPOCH:", BEST_EPOCH)
print("last common epoch:", LAST_COMMON_EPOCH)
print("mean temporal CV RMSLE:", f"{CV_RMSLE:.6f}")
print("CV std:", f"{CV_STD:.6f}")
print("January RMSLE at selected epoch:", f"{JAN_RMSLE:.6f}")

if BEST_EPOCH == LAST_COMMON_EPOCH:
    print(
        "WARNING: global optimum is still on the common-epoch boundary. "
        "Patience/max budget may still be too small."
    )


FOLD 1/3 | valid=2025-11-15
train: ['2025-04-19', '2025-05-19', '2025-06-18', '2025-07-18', '2025-08-17', '2025-09-16', '2025-10-16']
RESUME: epoch 1 completed -> start from 2
epoch 02/30 | lr=3.46e-04 | train_RMSE=1.7403 | valid_RMSLE=1.734352 | direct=1.738811 | hurdle=1.735883 | gate_BCE=0.4778 | AUC=0.8510 | p_mean=0.570/0.569 | mix=0.566 | no_improve=0/8
epoch 03/30 | lr=3.42e-04 | train_RMSE=1.7403 | valid_RMSLE=1.733294 | direct=1.734657 | hurdle=1.736653 | gate_BCE=0.4797 | AUC=0.8508 | p_mean=0.581/0.569 | mix=0.547 | no_improve=0/8
epoch 04/30 | lr=3.36e-04 | train_RMSE=1.7383 | valid_RMSLE=1.734345 | direct=1.739690 | hurdle=1.734147 | gate_BCE=0.4770 | AUC=0.8510 | p_mean=0.567/0.569 | mix=0.533 | no_improve=1/8
epoch 05/30 | lr=3.28e-04 | train_RMSE=1.7367 | valid_RMSLE=1.731818 | direct=1.736168 | hurdle=1.733806 | gate_BCE=0.4769 | AUC=0.8514 | p_mean=0.564/0.569 | mix=0.526 | no_improve=0/8
epoch 06/30 | lr=3.18e-04 | train_RMSE=1.7357 | valid_RMSLE=1.734990 | direct=1

## Learning curves и diagnostics

Individual fold curves показываю полностью.

Mean curves считаю **только по common epochs**, которые успели пройти все три fold.

Иначе поздняя эпоха одного fold выглядела бы как улучшение mean CV, хотя два других fold уже остановились.

In [ ]:
common_epochs = sorted(
    cv_summary["epoch"].unique()
)

common_df = cv_df[
    cv_df["epoch"].isin(common_epochs)
].copy()


# RMSLE по temporal fold.
fig, ax = plt.subplots(figsize=(9, 5))

for cutoff, part in cv_df.groupby("valid_cutoff"):
    part = part.sort_values("epoch")

    ax.plot(
        part["epoch"],
        part["rmsle"],
        marker="o",
        label=cutoff,
    )

mean_curve = (
    common_df.groupby("epoch")["rmsle"]
    .mean()
    .sort_index()
)

ax.plot(
    mean_curve.index,
    mean_curve.values,
    marker="o",
    linewidth=3,
    label="mean -- common epochs",
)

ax.axvline(
    BEST_EPOCH,
    linestyle="--",
    label=f"best epoch={BEST_EPOCH}",
)

ax.set_xlabel("Epoch")
ax.set_ylabel("RMSLE")
ax.set_title("Temporal validation RMSLE")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()

fig.savefig(
    MODEL_DIR / "cv_rmsle_curve.png",
    dpi=160,
    bbox_inches="tight",
)

plt.show()


# Train vs validation.
fig, ax = plt.subplots(figsize=(9, 5))

train_mean = (
    common_df.groupby("epoch")["train_rmsle_proxy"]
    .mean()
    .sort_index()
)

valid_mean = (
    common_df.groupby("epoch")["rmsle"]
    .mean()
    .sort_index()
)

ax.plot(
    train_mean.index,
    train_mean.values,
    marker="o",
    label="train sqrt(main MSE)",
)

ax.plot(
    valid_mean.index,
    valid_mean.values,
    marker="o",
    label="validation RMSLE",
)

ax.axvline(
    BEST_EPOCH,
    linestyle="--",
)

ax.set_xlabel("Epoch")
ax.set_ylabel("log-space RMSE")
ax.set_title("Train / validation gap")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()

fig.savefig(
    MODEL_DIR / "train_validation_gap.png",
    dpi=160,
    bbox_inches="tight",
)

plt.show()


# Direct vs hurdle vs blend.
fig, ax = plt.subplots(figsize=(9, 5))

curve = (
    common_df.groupby("epoch")[
        ["rmsle", "direct_rmsle", "hurdle_rmsle"]
    ]
    .mean()
    .sort_index()
)

ax.plot(
    curve.index,
    curve["rmsle"],
    marker="o",
    label="learned blend",
)

ax.plot(
    curve.index,
    curve["direct_rmsle"],
    marker="o",
    label="direct",
)

ax.plot(
    curve.index,
    curve["hurdle_rmsle"],
    marker="o",
    label="hurdle",
)

ax.axvline(
    BEST_EPOCH,
    linestyle="--",
)

ax.set_xlabel("Epoch")
ax.set_ylabel("Mean temporal RMSLE")
ax.set_title("Direct vs hurdle vs learned blend")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()

fig.savefig(
    MODEL_DIR / "direct_hurdle_blend_curve.png",
    dpi=160,
    bbox_inches="tight",
)

plt.show()


# Gate BCE/AUC.
fig, ax = plt.subplots(figsize=(9, 5))

gate_curve = (
    common_df.groupby("epoch")[
        ["gate_bce", "gate_auc"]
    ]
    .mean()
    .sort_index()
)

line1 = ax.plot(
    gate_curve.index,
    gate_curve["gate_bce"],
    marker="o",
    label="gate BCE",
)

ax.set_xlabel("Epoch")
ax.set_ylabel("BCE")

ax2 = ax.twinx()

line2 = ax2.plot(
    gate_curve.index,
    gate_curve["gate_auc"],
    marker="o",
    label="gate ROC-AUC",
)

ax2.set_ylabel("ROC-AUC")
ax.set_title("Gate learning")
ax.grid(alpha=0.25)

lines = line1 + line2

ax.legend(
    lines,
    [line.get_label() for line in lines],
    loc="best",
)

plt.tight_layout()

fig.savefig(
    MODEL_DIR / "gate_learning_curve.png",
    dpi=160,
    bbox_inches="tight",
)

plt.show()

## Error analysis на January fold

January -- последний out-of-time fold.

Смотрю:

- `true vs pred` в `log1p`;
- residual distribution;
- RMSLE отдельно для нулей и positive-квинтилей;
- calibration gate;
- самые большие ошибки.

Главная проблема остается в верхнем positive tail -- крупный будущий GMV модель чаще недооценивает.

In [ ]:
best_pred = last_fold_prediction_store[BEST_EPOCH]

target_log = np.log1p(last_fold_y)
pred_log = np.clip(best_pred["pred_log"], 0, None)
pred_value = np.expm1(pred_log)

diag = pd.DataFrame({
    "user_id": last_fold_users,
    "y_true": last_fold_y,
    "y_pred": pred_value,
    "target_log": target_log,
    "pred_log": pred_log,
    "residual_log": target_log - pred_log,
    "sq_error_log": (target_log - pred_log) ** 2,
    "gate_prob": best_pred["gate_prob"],
    "positive_log": best_pred["positive_log"],
    "direct_log": best_pred["direct_log"],
    "hurdle_log": best_pred["hurdle_log"],
})
diag.to_csv(MODEL_DIR / "jan_holdout_diagnostics.csv", index=False)

# True vs pred в log1p.
fig, ax = plt.subplots(figsize=(7, 6))
ax.hexbin(diag["target_log"], diag["pred_log"], gridsize=60, mincnt=1)
limit = float(max(
    diag["target_log"].quantile(0.995),
    diag["pred_log"].quantile(0.995),
))
ax.plot([0, limit], [0, limit], linestyle="--")
ax.set_xlim(0, limit)
ax.set_ylim(0, limit)
ax.set_xlabel("True log1p(GMV)")
ax.set_ylabel("Predicted log1p(GMV)")
ax.set_title(f"January holdout: true vs pred | RMSLE={JAN_RMSLE:.4f}")
plt.tight_layout()
fig.savefig(MODEL_DIR / "jan_true_vs_pred.png", dpi=160, bbox_inches="tight")
plt.show()

# Распределение residual.
fig, ax = plt.subplots(figsize=(9, 5))
clip = float(np.quantile(np.abs(diag["residual_log"]), 0.995))
ax.hist(diag["residual_log"].clip(-clip, clip), bins=80)
ax.axvline(0.0, linestyle="--")
ax.set_xlabel("true_log - pred_log")
ax.set_ylabel("Count")
ax.set_title("January holdout residuals")
ax.grid(alpha=0.20)
plt.tight_layout()
fig.savefig(MODEL_DIR / "jan_residual_hist.png", dpi=160, bbox_inches="tight")
plt.show()

# Нули + positive-квинтили.
diag["segment"] = "zero"
positive_idx = diag["y_true"] > 0
positive_rank = diag.loc[positive_idx, "target_log"].rank(method="first")
diag.loc[positive_idx, "segment"] = pd.qcut(
    positive_rank,
    q=5,
    labels=["positive_Q1", "positive_Q2", "positive_Q3", "positive_Q4", "positive_Q5"],
).astype(str)

segment_order = [
    "zero",
    "positive_Q1",
    "positive_Q2",
    "positive_Q3",
    "positive_Q4",
    "positive_Q5",
]

segment_rows = []
for segment in segment_order:
    part = diag[diag["segment"] == segment]
    if len(part) == 0:
        continue
    segment_rows.append({
        "segment": segment,
        "n": len(part),
        "share": len(part) / len(diag),
        "rmsle": float(np.sqrt(part["sq_error_log"].mean())),
        "mean_true_log": float(part["target_log"].mean()),
        "mean_pred_log": float(part["pred_log"].mean()),
        "mean_residual": float(part["residual_log"].mean()),
    })

segment_df = pd.DataFrame(segment_rows)
display(segment_df)

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(segment_df["segment"], segment_df["rmsle"])
ax.set_ylabel("RMSLE")
ax.set_title("January holdout error by target segment")
ax.tick_params(axis="x", rotation=30)
ax.grid(axis="y", alpha=0.20)
plt.tight_layout()
fig.savefig(MODEL_DIR / "jan_segment_rmsle.png", dpi=160, bbox_inches="tight")
plt.show()

# Calibration gate.
calibration = pd.DataFrame({
    "p": diag["gate_prob"],
    "nonzero": (diag["y_true"] > 0).astype(float),
})
calibration["bin"] = pd.cut(
    calibration["p"],
    bins=np.linspace(0, 1, 11),
    include_lowest=True,
)
calibration_df = (
    calibration.groupby("bin", observed=True)
    .agg(
        mean_pred=("p", "mean"),
        actual_rate=("nonzero", "mean"),
        n=("nonzero", "size"),
    )
    .reset_index()
)
display(calibration_df)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(calibration_df["mean_pred"], calibration_df["actual_rate"], marker="o")
ax.plot([0, 1], [0, 1], linestyle="--")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("Mean predicted P(y>0)")
ax.set_ylabel("Observed nonzero rate")
ax.set_title("January gate calibration")
ax.grid(alpha=0.25)
plt.tight_layout()
fig.savefig(MODEL_DIR / "jan_gate_calibration.png", dpi=160, bbox_inches="tight")
plt.show()

worst_errors = (
    diag.assign(abs_error=lambda x: np.abs(x["residual_log"]))
    .sort_values("abs_error", ascending=False)
    .head(30)
    [[
        "user_id",
        "y_true",
        "y_pred",
        "target_log",
        "pred_log",
        "residual_log",
        "gate_prob",
        "positive_log",
        "direct_log",
        "hurdle_log",
    ]]
)
display(worst_errors)

## Optional -- January gate oracle Димаса **(LSTM в любом виде крашит мое ядро, нужно будет отдельно его сделать, ДИмас, прости)**

Пока **полностью выключен**:

```python
RUN_JAN_GATE_ORACLE = False
```

Основной run -- только LSTM.

LightGBM-клетки ниже остаются в notebook как отдельный exploratory experiment, но при обычном запуске ничего не импортируют и ничего не обучают.

Сначала:

1. temporal CV LSTM;
2. early stopping выбирает `BEST_EPOCH`;
3. final train четырех LSTM seed;
4. ensemble четырех `pred_log`.

К oracle возвращаемся только отдельным экспериментом после основного LSTM run.

In [ ]:
RUN_JAN_GATE_ORACLE = False

if not RUN_JAN_GATE_ORACLE:
    print("January gate oracle: SKIPPED")
else:
    import gc
    import importlib.util
    import time

    import numpy as np
    import pandas as pd

    if importlib.util.find_spec("lightgbm") is None:
        raise ImportError(
            "Установите зависимости:\n"
            "pip install -r notebooks/modeling/two_stage_v2/requirements-v2.txt"
        )

    import lightgbm as lgb
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
    from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit


    ORACLE_SEED = 2026
    ORACLE_OUTER_FOLDS = 5

    # Насколько сильно разрешаем LightGBM менять вероятность LSTM.
    ORACLE_LAMBDA_GRID = np.linspace(0.0, 1.0, 11)

    # Компактная модель: это тест гипотезы, не финальный тюнинг.
    ORACLE_MAX_TREES = 1500
    ORACLE_EARLY_STOPPING_ROUNDS = 100
    ORACLE_N_JOBS = -1

    PROB_EPS = 1e-6


    def clip_probability(probability):
        return np.clip(
            np.asarray(probability, dtype=np.float64),
            PROB_EPS,
            1.0 - PROB_EPS,
        )


    def probability_logit(probability):
        probability = clip_probability(probability)
        return np.log(probability) - np.log1p(-probability)


    def probability_sigmoid(logit):
        logit = np.clip(np.asarray(logit, dtype=np.float64), -40.0, 40.0)
        return 1.0 / (1.0 + np.exp(-logit))


    def fit_platt(probability, target, seed):
        """
        Обычная Platt-калибровка:
        sigmoid(a * logit(p) + b).
        """
        target = np.asarray(target, dtype=np.int8)

        if np.unique(target).size != 2:
            raise ValueError("Platt calibration needs both target classes")

        calibrator = LogisticRegression(
            C=10.0,
            solver="lbfgs",
            max_iter=1000,
            random_state=seed,
        )
        calibrator.fit(
            probability_logit(probability).reshape(-1, 1),
            target,
        )
        return calibrator


    def apply_platt(calibrator, probability):
        return calibrator.predict_proba(
            probability_logit(probability).reshape(-1, 1)
        )[:, 1]


    def blend_probabilities_logit(
        lstm_probability,
        oracle_probability,
        weight,
    ):
        """
        weight=0 -> только LSTM.
        weight=1 -> только LightGBM.
    
        Смешиваем logits, а не сами вероятности:
        это устойчивее около 0 и 1.
        """
        if not 0.0 <= weight <= 1.0:
            raise ValueError("weight must be in [0, 1]")

        blended_logit = (
            (1.0 - weight) * probability_logit(lstm_probability)
            + weight * probability_logit(oracle_probability)
        )
        return probability_sigmoid(blended_logit)


    def optimal_alpha(target_log, hurdle_log, direct_log):
        """
        Аналитически находит лучший вес:

            prediction =
                alpha * hurdle_log
                + (1 - alpha) * direct_log

        Так как RMSLE — это RMSE в log1p-space,
        alpha можно найти как решение одномерной least-squares задачи.
        """
        target_log = np.asarray(target_log, dtype=np.float64)
        hurdle_log = np.asarray(hurdle_log, dtype=np.float64)
        direct_log = np.asarray(direct_log, dtype=np.float64)

        direction = hurdle_log - direct_log
        denominator = float(direction @ direction)

        if denominator <= 1e-12:
            return 0.0

        alpha = float(
            direction @ (target_log - direct_log)
            / denominator
        )
        return float(np.clip(alpha, 0.0, 1.0))


    def compose_log_prediction(
        gate_probability,
        positive_log,
        direct_log,
        alpha,
    ):
        """
        1. gate_probability определяет вероятность ненулевого месяца.
        2. positive_log оценивает GMV при условии активности.
        3. direct_log — независимый regression head.
        4. alpha смешивает hurdle и direct predictions.
        """
        if not 0.0 <= alpha <= 1.0:
            raise ValueError("alpha must be in [0, 1]")

        gate_probability = np.asarray(
            gate_probability, dtype=np.float64
        )
        positive_log = np.asarray(positive_log, dtype=np.float64)
        direct_log = np.asarray(direct_log, dtype=np.float64)

        if not (
            gate_probability.shape
            == positive_log.shape
            == direct_log.shape
        ):
            raise ValueError("gate/head arrays must have equal shapes")

        values = np.column_stack([
            gate_probability,
            positive_log,
            direct_log,
        ])
        if not np.isfinite(values).all():
            raise ValueError("gate/head arrays must be finite")

        if np.any(
            (gate_probability < 0.0)
            | (gate_probability > 1.0)
        ):
            raise ValueError("gate_probability must be in [0, 1]")

        hurdle_log = gate_probability * positive_log

        prediction = (
            alpha * hurdle_log
            + (1.0 - alpha) * direct_log
        )
        return np.clip(prediction, 0.0, None)


    def rmsle_from_logs(target_log, prediction_log):
        error = (
            np.asarray(target_log)
            - np.asarray(prediction_log)
        )
        return float(np.sqrt(np.mean(error * error)))


    def select_lambda_and_alpha(
        lstm_probability,
        oracle_probability,
        positive_log,
        direct_log,
        target_log,
        lambda_grid,
    ):
        """
        На отдельной tuning-выборке совместно выбирает:
        - lambda: насколько доверять LightGBM;
        - alpha: насколько доверять hurdle-head.
        """
        candidates = []

        for weight in lambda_grid:
            blended_probability = blend_probabilities_logit(
                lstm_probability,
                oracle_probability,
                float(weight),
            )

            hurdle_log = blended_probability * positive_log
            alpha = optimal_alpha(
                target_log,
                hurdle_log,
                direct_log,
            )

            prediction_log = compose_log_prediction(
                blended_probability,
                positive_log,
                direct_log,
                alpha,
            )

            candidates.append({
                "lambda": float(weight),
                "alpha": alpha,
                "rmsle": rmsle_from_logs(
                    target_log,
                    prediction_log,
                ),
            })

        # При равном качестве предпочитаем меньше вмешиваться в LSTM.
        return min(
            candidates,
            key=lambda row: (row["rmsle"], row["lambda"]),
        )


    def stratified_split(
        indices,
        target,
        test_size,
        seed,
    ):
        indices = np.asarray(indices)

        splitter = StratifiedShuffleSplit(
            n_splits=1,
            test_size=test_size,
            random_state=seed,
        )

        train_local, test_local = next(
            splitter.split(
                np.zeros(len(indices), dtype=np.int8),
                target[indices],
            )
        )

        return indices[train_local], indices[test_local]


    def make_oracle_model(seed, n_estimators):
        return lgb.LGBMClassifier(
            objective="binary",
            n_estimators=n_estimators,
            learning_rate=0.03,

            # Небольшие деревья снижают риск запоминания января.
            num_leaves=15,
            max_depth=4,
            min_child_samples=1000,

            subsample=0.80,
            subsample_freq=1,
            colsample_bytree=0.75,

            reg_alpha=0.5,
            reg_lambda=2.0,
            max_bin=127,

            random_state=seed,
            n_jobs=ORACLE_N_JOBS,
            verbosity=-1,

            deterministic=True,
            force_col_wise=True,
        )


    # Мини-тесты самых опасных формул.
    _test_lstm = np.array([0.2, 0.8])
    _test_oracle = np.array([0.7, 0.3])

    np.testing.assert_allclose(
        blend_probabilities_logit(
            _test_lstm, _test_oracle, 0.0
        ),
        _test_lstm,
    )
    np.testing.assert_allclose(
        blend_probabilities_logit(
            _test_lstm, _test_oracle, 1.0
        ),
        _test_oracle,
    )

    _test_hurdle = np.array([0.0, 2.0, 4.0, 6.0])
    _test_direct = np.ones(4)
    _test_target = (
        0.75 * _test_hurdle
        + 0.25 * _test_direct
    )

    assert abs(
        optimal_alpha(
            _test_target,
            _test_hurdle,
            _test_direct,
        )
        - 0.75
    ) < 1e-12

    print("Oracle helper checks: OK")

    # Standalone bootstrap: this oracle cell can run in a clean kernel.
    from pathlib import Path
    import json


    def find_oracle_project_root(start=None):
        start = Path.cwd() if start is None else Path(start).resolve()
        for candidate in [start, *start.parents]:
            if (
                (candidate / "data" / "lstm" / "meta.json").exists()
                and (
                    candidate
                    / "models"
                    / "lstm_hurdle_v4_robust"
                    / "jan_holdout_diagnostics.csv"
                ).exists()
            ):
                return candidate
        raise FileNotFoundError(
            "Не найден корень проекта с data/lstm/meta.json и January diagnostics"
        )


    PROJECT_ROOT = find_oracle_project_root()
    DATA_DIR = PROJECT_ROOT / "data" / "lstm"
    MODEL_DIR = PROJECT_ROOT / "models" / "lstm_hurdle_v4_robust"

    with open(DATA_DIR / "meta.json", encoding="utf-8") as file:
        META = json.load(file)

    if META.get("format_version") != 2:
        raise RuntimeError("Oracle expects data/lstm format_version=2")

    SEQ_LEN = int(META["seq_len"])
    diag_path = MODEL_DIR / "jan_holdout_diagnostics.csv"

    if "diag" not in globals():
        diag = pd.read_csv(diag_path)
    else:
        diag = diag.copy()

    diag = diag.reset_index(drop=True)


    required_diagnostic_columns = {
        "user_id",
        "y_true",
        "target_log",
        "pred_log",
        "gate_prob",
        "positive_log",
        "direct_log",
        "hurdle_log",
    }

    missing_columns = (
        required_diagnostic_columns
        - set(diag.columns)
    )

    if missing_columns:
        raise KeyError(
            f"Missing diagnostic columns: {sorted(missing_columns)}"
        )


    # Проверяем, что diagnostic и static.npy относятся
    # к одним пользователям и стоят в одном порядке.
    jan_cutoff = META["labeled_cutoffs"][-1]
    jan_dir = DATA_DIR / jan_cutoff

    jan_users = np.load(jan_dir / "user_id.npy")
    diag_users = diag["user_id"].to_numpy(
        dtype=jan_users.dtype,
        copy=False,
    )

    assert not diag["user_id"].isna().any()
    assert diag["user_id"].is_unique
    assert len(diag) == len(jan_users)
    assert np.array_equal(diag_users, jan_users)


    user_index_path = jan_dir / "user_index.npy"
    all_users_path = DATA_DIR / "all_user_ids.npy"

    if user_index_path.exists() and all_users_path.exists():
        user_index = np.load(user_index_path)
        all_users = np.load(all_users_path)

        assert np.array_equal(
            all_users[user_index],
            jan_users,
        )


    raw_static = np.load(
        jan_dir / "static.npy",
        mmap_mode="r",
    )
    history_length = np.load(
        jan_dir / "history_length.npy"
    )

    raw_static_names = list(META["static_features"])

    assert raw_static.shape[0] == len(diag)
    assert raw_static.shape[1] == len(raw_static_names)
    assert len(raw_static_names) == len(set(raw_static_names))
    assert len(history_length) == len(diag)
    assert np.all(
        (history_length >= 1)
        & (history_length <= SEQ_LEN)
    )


    # Компактный набор, чтобы первый эксперимент был быстрым.
    # Он покрывает recency, frequency, monetary value,
    # engagement, тренды и признаки необычных клиентов.
    oracle_static_names = [
        "customer_age_days",
        "never_purchased",
        "days_since_last_purchase",
        "days_since_last_activity",
        "days_since_last_search",
        "days_since_last_cart",

        "gmv_7d",
        "gmv_30d",
        "gmv_90d",

        "purchased_items_7d",
        "purchased_items_30d",
        "purchased_items_90d",

        "searches_7d",
        "searches_30d",
        "searches_90d",

        "active_days_30d",
        "active_days_90d",

        "gmv_trend_log",
        "intent_trend_log",
        "whale_score",

        "purchase_frequency",
        "recency_ratio",
        "active_day_rate_lifetime",
        "no_recent_engagement",

        "gmv_per_item",
        "gmv_daily_mean",
        "search_to_purchase_freq",

        "searches_trend_log",
        "purchased_items_trend_log",
        "active_days_trend_log",
    ]

    missing_static = sorted(
        set(oracle_static_names)
        - set(raw_static_names)
    )

    if missing_static:
        raise KeyError(
            f"Static features missing from metadata: {missing_static}"
        )


    # Frozen LSTM outputs.
    lstm_gate = clip_probability(
        diag["gate_prob"].to_numpy()
    )
    positive_log = diag[
        "positive_log"
    ].to_numpy(dtype=np.float64)
    direct_log = diag[
        "direct_log"
    ].to_numpy(dtype=np.float64)
    hurdle_log = diag[
        "hurdle_log"
    ].to_numpy(dtype=np.float64)
    original_pred_log = diag[
        "pred_log"
    ].to_numpy(dtype=np.float64)

    # Эти два массива используются только как labels/evaluation,
    # они не попадут в X_oracle.
    target_log = diag[
        "target_log"
    ].to_numpy(dtype=np.float64)
    target_active = (
        diag["y_true"].to_numpy() > 0
    ).astype(np.int8)


    assert np.unique(target_active).size == 2
    assert np.isfinite(
        np.column_stack([
            lstm_gate,
            positive_log,
            direct_log,
            hurdle_log,
            original_pred_log,
            target_log,
        ])
    ).all()

    assert (positive_log >= 0).all()
    assert (direct_log >= 0).all()


    # Признаки, описывающие мнение самой LSTM
    # и внутренние противоречия между её heads.
    meta_columns = {
        "lstm_gate_prob":
            lstm_gate.astype(np.float32),

        "lstm_gate_logit":
            probability_logit(lstm_gate).astype(np.float32),

        # Максимум при p=0.5 — зона неуверенности.
        "lstm_gate_uncertainty":
            (lstm_gate * (1.0 - lstm_gate)).astype(np.float32),

        "lstm_gate_entropy": (
            -lstm_gate * np.log(lstm_gate)
            - (1.0 - lstm_gate) * np.log1p(-lstm_gate)
        ).astype(np.float32),

        "lstm_pred_log":
            original_pred_log.astype(np.float32),

        "lstm_positive_log":
            positive_log.astype(np.float32),

        "lstm_direct_log":
            direct_log.astype(np.float32),

        "lstm_hurdle_log":
            hurdle_log.astype(np.float32),

        "positive_minus_direct":
            (positive_log - direct_log).astype(np.float32),

        "hurdle_minus_direct":
            (hurdle_log - direct_log).astype(np.float32),

        "pred_minus_direct":
            (original_pred_log - direct_log).astype(np.float32),

        "pred_minus_hurdle":
            (original_pred_log - hurdle_log).astype(np.float32),

        "history_length":
            history_length.astype(np.float32),
    }


    # Важно: извлекаем static по именам, а не по предполагаемым позициям.
    for feature_name in oracle_static_names:
        column_index = raw_static_names.index(feature_name)

        meta_columns[feature_name] = np.asarray(
            raw_static[:, column_index],
            dtype=np.float32,
        )


    X_oracle = (
        pd.DataFrame(meta_columns)
        .replace([np.inf, -np.inf], np.nan)
    )


    # Защита от случайного попадания target/identity.
    forbidden_tokens = (
        "target",
        "residual",
        "sq_error",
        "y_true",
        "user_id",
    )

    assert not any(
        token in column.lower()
        for column in X_oracle.columns
        for token in forbidden_tokens
    )

    model_output_columns = [
        column
        for column in X_oracle.columns
        if column.startswith("lstm_")
    ]

    assert np.isfinite(
        X_oracle[model_output_columns].to_numpy()
    ).all()


    print("January rows:", len(X_oracle))
    print("Oracle features:", X_oracle.shape[1])
    print(
        "Positive class share:",
        f"{target_active.mean():.3%}",
    )
    print(
        "Meta-table memory:",
        f"{X_oracle.memory_usage(deep=True).sum() / 2**20:.1f} MiB",
    )

lambda отвечает за силу вмешательства LightGBM, alpha -- за новый баланс между hurdle и direct heads. Их нельзя подбирать на той же выборке, на которой мы измеряем итоговый RMSLE иначе будет утечка данных

# Построение meta-таблицы

В X_oracle принципиально нет:
- user_id;
- y_true;
- target_log;
- residual_log;
- sq_error_log.
LightGBM видит только информацию, доступную на момент прогноза.

# Nested cross-fitting оракула

In [ ]:
if not RUN_JAN_GATE_ORACLE:
    print("January gate oracle block: SKIPPED")
else:
    n_rows = len(X_oracle)

    oof_platt_gate = np.full(
        n_rows, np.nan, dtype=np.float64
    )
    oof_meta_gate = np.full(
        n_rows, np.nan, dtype=np.float64
    )
    oof_oracle_gate = np.full(
        n_rows, np.nan, dtype=np.float64
    )

    oof_platt_pred_log = np.full(
        n_rows, np.nan, dtype=np.float64
    )
    oof_oracle_pred_log = np.full(
        n_rows, np.nan, dtype=np.float64
    )

    fold_rows = []
    feature_gain = np.zeros(
        X_oracle.shape[1],
        dtype=np.float64,
    )


    outer_cv = StratifiedKFold(
        n_splits=ORACLE_OUTER_FOLDS,
        shuffle=True,
        random_state=ORACLE_SEED,
    )


    for fold, (outer_train, outer_test) in enumerate(
        outer_cv.split(X_oracle, target_active),
        start=1,
    ):
        started = time.time()
        seed = ORACLE_SEED + fold

        # 1. tuning: только lambda и alpha.
        development, tuning = stratified_split(
            outer_train,
            target_active,
            test_size=0.20,
            seed=seed,
        )

        # 2. calibration: только Platt.
        core, calibration = stratified_split(
            development,
            target_active,
            test_size=0.20,
            seed=seed + 100,
        )

        # 3. early_stop: только выбор числа деревьев.
        fit_rows, early_stop = stratified_split(
            core,
            target_active,
            test_size=0.20,
            seed=seed + 200,
        )

        # 4. outer_test не участвует ни в одном выборе.
        split_arrays = [
            fit_rows,
            early_stop,
            calibration,
            tuning,
            outer_test,
        ]

        combined_rows = np.concatenate(split_arrays)

        assert len(combined_rows) == n_rows
        assert np.unique(combined_rows).size == n_rows
        assert all(
            np.unique(target_active[rows]).size == 2
            for rows in split_arrays
        )

        # Сначала маленький probe-run для выбора числа деревьев.
        probe = make_oracle_model(
            seed,
            ORACLE_MAX_TREES,
        )

        probe.fit(
            X_oracle.iloc[fit_rows],
            target_active[fit_rows],

            eval_set=[(
                X_oracle.iloc[early_stop],
                target_active[early_stop],
            )],
            eval_metric="binary_logloss",

            callbacks=[
                lgb.early_stopping(
                    ORACLE_EARLY_STOPPING_ROUNDS,
                    verbose=False,
                ),
                lgb.log_evaluation(0),
            ],
        )

        best_iteration = int(
            probe.best_iteration_
            or ORACLE_MAX_TREES
        )

        # После выбора количества деревьев переобучаем
        # только LightGBM на fit + early_stop.
        oracle = make_oracle_model(
            seed,
            best_iteration,
        )
        oracle.fit(
            X_oracle.iloc[core],
            target_active[core],
        )

        # LightGBM probability до калибровки.
        meta_calibration_raw = oracle.predict_proba(
            X_oracle.iloc[calibration]
        )[:, 1]

        meta_tuning_raw = oracle.predict_proba(
            X_oracle.iloc[tuning]
        )[:, 1]

        meta_test_raw = oracle.predict_proba(
            X_oracle.iloc[outer_test]
        )[:, 1]

        # Platt поверх LightGBM.
        meta_platt = fit_platt(
            meta_calibration_raw,
            target_active[calibration],
            seed,
        )

        meta_tuning = apply_platt(
            meta_platt,
            meta_tuning_raw,
        )
        meta_test = apply_platt(
            meta_platt,
            meta_test_raw,
        )

        # Отдельный дешёвый baseline:
        # что даст одна калибровка LSTM без LightGBM?
        lstm_platt = fit_platt(
            lstm_gate[calibration],
            target_active[calibration],
            seed,
        )

        lstm_platt_tuning = apply_platt(
            lstm_platt,
            lstm_gate[tuning],
        )
        lstm_platt_test = apply_platt(
            lstm_platt,
            lstm_gate[outer_test],
        )

        platt_choice = select_lambda_and_alpha(
            lstm_probability=lstm_gate[tuning],
            oracle_probability=lstm_platt_tuning,
            positive_log=positive_log[tuning],
            direct_log=direct_log[tuning],
            target_log=target_log[tuning],

            # Здесь lambda=1 означает:
            # полностью заменить gate его Platt-версией.
            lambda_grid=np.array([1.0]),
        )

        platt_test_pred = compose_log_prediction(
            gate_probability=lstm_platt_test,
            positive_log=positive_log[outer_test],
            direct_log=direct_log[outer_test],
            alpha=platt_choice["alpha"],
        )

        # Выбираем силу вмешательства LightGBM.
        oracle_choice = select_lambda_and_alpha(
            lstm_probability=lstm_gate[tuning],
            oracle_probability=meta_tuning,
            positive_log=positive_log[tuning],
            direct_log=direct_log[tuning],
            target_log=target_log[tuning],
            lambda_grid=ORACLE_LAMBDA_GRID,
        )

        oracle_test_gate = blend_probabilities_logit(
            lstm_gate[outer_test],
            meta_test,
            oracle_choice["lambda"],
        )

        oracle_test_pred = compose_log_prediction(
            gate_probability=oracle_test_gate,
            positive_log=positive_log[outer_test],
            direct_log=direct_log[outer_test],
            alpha=oracle_choice["alpha"],
        )

        # Каждый пользователь получает предсказание модели,
        # которая не видела его target.
        oof_platt_gate[outer_test] = lstm_platt_test
        oof_meta_gate[outer_test] = meta_test
        oof_oracle_gate[outer_test] = oracle_test_gate

        oof_platt_pred_log[outer_test] = platt_test_pred
        oof_oracle_pred_log[outer_test] = oracle_test_pred

        # Нормализованный gain importance.
        gain = oracle.booster_.feature_importance(
            importance_type="gain"
        )

        if gain.sum() > 0:
            feature_gain += gain / gain.sum()

        original_fold_rmsle = rmsle_from_logs(
            target_log[outer_test],
            original_pred_log[outer_test],
        )
        platt_fold_rmsle = rmsle_from_logs(
            target_log[outer_test],
            platt_test_pred,
        )
        oracle_fold_rmsle = rmsle_from_logs(
            target_log[outer_test],
            oracle_test_pred,
        )

        fold_rows.append({
            "fold": fold,
            "n_test": len(outer_test),
            "n_fit": len(fit_rows),
            "n_early_stop": len(early_stop),
            "n_calibration": len(calibration),
            "n_tuning": len(tuning),

            "best_iteration": best_iteration,
            "lambda": oracle_choice["lambda"],
            "alpha_platt": platt_choice["alpha"],
            "alpha_oracle": oracle_choice["alpha"],

            "rmsle_original": original_fold_rmsle,
            "rmsle_platt": platt_fold_rmsle,
            "rmsle_oracle": oracle_fold_rmsle,

            "delta_oracle": (
                oracle_fold_rmsle
                - original_fold_rmsle
            ),
            "seconds": time.time() - started,
        })

        print(
            f"fold={fold} "
            f"trees={best_iteration} "
            f"lambda={oracle_choice['lambda']:.2f} "
            f"alpha={oracle_choice['alpha']:.3f} | "
            f"RMSLE {original_fold_rmsle:.6f} "
            f"-> {oracle_fold_rmsle:.6f} "
            f"({oracle_fold_rmsle - original_fold_rmsle:+.6f})"
        )

        del probe, oracle, meta_platt, lstm_platt
        gc.collect()


    for output in [
        oof_platt_gate,
        oof_meta_gate,
        oof_oracle_gate,
        oof_platt_pred_log,
        oof_oracle_pred_log,
    ]:
        assert np.isfinite(output).all(), (
            "OOF array contains unfilled rows"
        )


    fold_results = pd.DataFrame(fold_rows)
    display(fold_results)

Внутри каждого outer-fold находятся четыре непересекающиеся роли:

```
outer train
├── fit          → обучение LightGBM
├── early_stop   → количество деревьев
├── calibration  → Platt
└── tuning       → lambda и alpha

outer test       → только итоговая оценка
```

# Проверка гипотезы

In [ ]:
if not RUN_JAN_GATE_ORACLE:
    print("January gate oracle block: SKIPPED")
else:
    positive_mask = target_active == 1

    positive_q5_threshold = np.quantile(
        target_log[positive_mask],
        0.80,
    )
    positive_q5_mask = (
        positive_mask
        & (target_log >= positive_q5_threshold)
    )


    def evaluation_row(
        name,
        gate_probability,
        prediction_log,
    ):
        probability = clip_probability(
            gate_probability
        )

        rmsle = rmsle_from_logs(
            target_log,
            prediction_log,
        )
        original_rmsle = rmsle_from_logs(
            target_log,
            original_pred_log,
        )

        return {
            "model": name,
            "RMSLE": rmsle,
            "delta_vs_original": rmsle - original_rmsle,

            "gate_logloss": log_loss(
                target_active,
                probability,
                labels=[0, 1],
            ),
            "gate_AUC": roc_auc_score(
                target_active,
                probability,
            ),
            "gate_Brier": brier_score_loss(
                target_active,
                probability,
            ),

            "RMSLE_zero": rmsle_from_logs(
                target_log[~positive_mask],
                prediction_log[~positive_mask],
            ),
            "RMSLE_positive": rmsle_from_logs(
                target_log[positive_mask],
                prediction_log[positive_mask],
            ),
            "RMSLE_positive_Q5": rmsle_from_logs(
                target_log[positive_q5_mask],
                prediction_log[positive_q5_mask],
            ),
        }


    oracle_report = pd.DataFrame([
        evaluation_row(
            "Original LSTM",
            lstm_gate,
            original_pred_log,
        ),
        evaluation_row(
            "Platt(LSTM gate) + tuned alpha",
            oof_platt_gate,
            oof_platt_pred_log,
        ),
        evaluation_row(
            "LightGBM oracle + logit blend",
            oof_oracle_gate,
            oof_oracle_pred_log,
        ),
    ])

    display(oracle_report)


    # Качество самого LightGBM-классификатора
    # до смешивания с LSTM.
    meta_gate_report = pd.DataFrame([
        {
            "gate": "Original LSTM",
            "logloss": log_loss(
                target_active,
                clip_probability(lstm_gate),
                labels=[0, 1],
            ),
            "AUC": roc_auc_score(
                target_active,
                lstm_gate,
            ),
            "Brier": brier_score_loss(
                target_active,
                lstm_gate,
            ),
        },
        {
            "gate": "LightGBM calibrated before blend",
            "logloss": log_loss(
                target_active,
                clip_probability(oof_meta_gate),
                labels=[0, 1],
            ),
            "AUC": roc_auc_score(
                target_active,
                oof_meta_gate,
            ),
            "Brier": brier_score_loss(
                target_active,
                oof_meta_gate,
            ),
        },
    ])

    display(meta_gate_report)


    feature_importance = (
        pd.DataFrame({
            "feature": X_oracle.columns,
            "mean_gain_share": (
                feature_gain
                / ORACLE_OUTER_FOLDS
            ),
        })
        .sort_values(
            "mean_gain_share",
            ascending=False,
            ignore_index=True,
        )
    )

    display(feature_importance.head(25))


    def paired_bootstrap_rmsle_delta(
        target_log,
        baseline_log,
        candidate_log,
        n_resamples=300,
        seed=ORACLE_SEED,
    ):
        """
        Быстрый paired bootstrap по пользователям.

        Для окончательного отчёта можно поднять
        n_resamples до 2000.
        """
        rng = np.random.default_rng(seed)

        baseline_sq = (
            target_log - baseline_log
        ) ** 2
        candidate_sq = (
            target_log - candidate_log
        ) ** 2

        deltas = np.empty(
            n_resamples,
            dtype=np.float64,
        )
        n = len(target_log)

        for iteration in range(n_resamples):
            sample = rng.integers(
                0,
                n,
                size=n,
            )

            deltas[iteration] = (
                np.sqrt(candidate_sq[sample].mean())
                - np.sqrt(baseline_sq[sample].mean())
            )

        return np.quantile(
            deltas,
            [0.025, 0.50, 0.975],
        )


    bootstrap_ci = paired_bootstrap_rmsle_delta(
        target_log,
        original_pred_log,
        oof_oracle_pred_log,
    )

    overall_delta = (
        rmsle_from_logs(
            target_log,
            oof_oracle_pred_log,
        )
        - rmsle_from_logs(
            target_log,
            original_pred_log,
        )
    )

    better_folds = int(
        (fold_results["delta_oracle"] < 0).sum()
    )


    original_row = (
        oracle_report
        .set_index("model")
        .loc["Original LSTM"]
    )
    oracle_row = (
        oracle_report
        .set_index("model")
        .loc["LightGBM oracle + logit blend"]
    )

    guardrails_pass = bool(
        oracle_row["gate_logloss"]
        <= original_row["gate_logloss"] + 0.01

        and oracle_row["gate_Brier"]
        <= original_row["gate_Brier"] + 0.01

        and oracle_row["RMSLE_zero"]
        <= original_row["RMSLE_zero"] + 0.02

        and oracle_row["RMSLE_positive_Q5"]
        <= original_row["RMSLE_positive_Q5"] + 0.02
    )


    # Диагностическая верхняя граница.
    # Здесь используется настоящий label — это НЕ модель.
    current_direction = hurdle_log - direct_log

    current_alpha = float(np.clip(
        current_direction
        @ (original_pred_log - direct_log)
        / max(
            float(current_direction @ current_direction),
            1e-12,
        ),
        0.0,
        1.0,
    ))

    perfect_gate = target_active.astype(np.float64)
    perfect_hurdle = perfect_gate * positive_log

    perfect_alpha = optimal_alpha(
        target_log,
        perfect_hurdle,
        direct_log,
    )

    perfect_current_pred = compose_log_prediction(
        perfect_gate,
        positive_log,
        direct_log,
        current_alpha,
    )

    perfect_optimal_pred = compose_log_prediction(
        perfect_gate,
        positive_log,
        direct_log,
        perfect_alpha,
    )


    print(
        "Paired bootstrap 95% CI for RMSLE delta:",
        bootstrap_ci,
    )
    print(
        f"Better outer folds: "
        f"{better_folds}/{ORACLE_OUTER_FOLDS}"
    )
    print("Guardrails passed:", guardrails_pass)

    print("\nLABEL-INFORMED DIAGNOSTIC UPPER BOUND")
    print(
        f"Recovered current alpha: "
        f"{current_alpha:.4f}"
    )
    print(
        "Perfect gate + current alpha RMSLE:",
        f"{rmsle_from_logs(target_log, perfect_current_pred):.6f}",
    )
    print(
        f"Perfect gate + optimal alpha={perfect_alpha:.4f}:",
        f"{rmsle_from_logs(target_log, perfect_optimal_pred):.6f}",
    )


    if (
        overall_delta <= -0.001
        and bootstrap_ci[2] <= 0
        and guardrails_pass
    ):
        print(
            "\nVERDICT: January exploratory-pass. "
            "Следующий шаг — собрать temporal LSTM OOF "
            "по нескольким cutoff и проверить перенос по месяцам."
        )
    elif overall_delta < 0:
        print(
            "\nVERDICT: слабый или смешанный положительный сигнал. "
            "Смотрите fold_results и segment-метрики; "
            "для submission доказательств пока недостаточно."
        )
    else:
        print(
            "\nVERDICT: вариант B не улучшил frozen LSTM "
            "на January random cross-fit."
        )

## Screening перед final train

Сравниваю новую training policy с текущей public-best v4.

Reference:

```text
mean temporal CV = 1.715288
January RMSLE    = 1.675888
```

Final train разрешаю, если новая версия не развалилась ни по mean CV, ни по January.

`FORCE_FINAL_TRAIN=True` все еще позволяет вручную переопределить screen.

In [ ]:
jan_limit = (
    REFERENCE_JAN_HOLDOUT_RMSLE
    + MAX_JAN_DEGRADATION_FOR_FINAL
)

cv_limit = (
    REFERENCE_MEAN_CV_RMSLE
    + MAX_MEAN_CV_DEGRADATION_FOR_FINAL
)

passes_jan_screen = (
    JAN_RMSLE <= jan_limit
)

passes_cv_screen = (
    CV_RMSLE <= cv_limit
)

last_common_mean = float(
    common_df[
        common_df["epoch"] == LAST_COMMON_EPOCH
    ]["rmsle"].mean()
)

overfit_from_best_to_last = (
    last_common_mean - CV_RMSLE
)

print(
    "reference mean CV RMSLE:",
    f"{REFERENCE_MEAN_CV_RMSLE:.6f}",
)

print(
    "new mean CV RMSLE:",
    f"{CV_RMSLE:.6f}",
)

print(
    "mean CV screen limit:",
    f"{cv_limit:.6f}",
)

print(
    "reference January RMSLE:",
    f"{REFERENCE_JAN_HOLDOUT_RMSLE:.6f}",
)

print(
    "new January RMSLE:",
    f"{JAN_RMSLE:.6f}",
)

print(
    "January screen limit:",
    f"{jan_limit:.6f}",
)

print(
    "best->last common validation delta:",
    f"{overfit_from_best_to_last:+.6f}",
)

if BEST_EPOCH == LAST_COMMON_EPOCH:
    print(
        "WARNING: selected epoch is still the last common epoch."
    )

if CV_STD > 0.04:
    print(
        "WARNING: fold-to-fold variance high:",
        f"{CV_STD:.4f}",
    )

SHOULD_RUN_FINAL = (
    FORCE_FINAL_TRAIN
    or not AUTO_SKIP_FINAL_IF_WEAK
    or (
        passes_jan_screen
        and passes_cv_screen
    )
)

print(
    "FINAL TRAIN:",
    "ENABLED"
    if SHOULD_RUN_FINAL
    else "SKIPPED",
)

## Финальное обучение

После CV обучаю **четыре seed** на всех labeled cutoff:

```text
42
143
67
2026
```

Final early stopping здесь не используется -- отдельного validation target уже нет.

Вместо этого все четыре модели обучаются ровно `BEST_EPOCH`, выбранный temporal CV.

Scheduler у всех тот же, что в CV:

```text
CosineAnnealingLR(T_max=MAX_CV_EPOCHS)
```

То есть training dynamics до `BEST_EPOCH` совпадает с тем, что валидировалось на CV.

Checkpoint каждого seed сохраняется после каждой эпохи.

In [ ]:
final_static_stats = fit_static_stats(
    LABELED_CUTOFFS
)

final_known_user_mask = (
    known_user_mask_from_cutoffs(
        LABELED_CUTOFFS
    )
)

final_models = []


FINAL_TRAIN_SIGNATURE = hashlib.sha256(
    json.dumps(
        {
            "architecture": "JointHurdleLSTM-v4-expanded-es",
            "model_kwargs": MODEL_KWARGS,
            "best_epoch": BEST_EPOCH,
            "scheduler_t_max": MAX_CV_EPOCHS,
            "lr": LR,
            "weight_decay": WEIGHT_DECAY,
            "aux_weights": {
                "gate": AUX_GATE_WEIGHT,
                "positive": AUX_POS_WEIGHT,
                "direct": AUX_DIRECT_WEIGHT,
            },
            "flags": {
                "user_embedding": USE_USER_EMBEDDING,
                "conv_stem": USE_CONV_STEM,
                "absolute_time": USE_ABSOLUTE_TIME,
                "future_calendar": USE_FUTURE_CALENDAR_STATIC,
                "cutoff_seasonal": USE_CUTOFF_SEASONAL_STATIC,
            },
        },
        sort_keys=True,
        default=str,
    ).encode("utf-8")
).hexdigest()[:16]


def _load_resume(path, map_location="cpu"):
    try:
        return torch.load(
            path,
            map_location=map_location,
            weights_only=False,
        )
    except TypeError:
        return torch.load(
            path,
            map_location=map_location,
        )


def _save_resume_checkpoint(
    path,
    seed,
    epoch,
    model,
    optimizer,
    scheduler,
    scaler,
    history,
):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    payload = {
        "architecture": "JointHurdleLSTM-v4-expanded-es",
        "training_signature": FINAL_TRAIN_SIGNATURE,
        "seed": int(seed),
        "epoch": int(epoch),
        "best_epoch": int(BEST_EPOCH),
        "scheduler_t_max": int(MAX_CV_EPOCHS),
        "model_kwargs": MODEL_KWARGS,
        "state_dict": {
            key: value.detach().cpu()
            for key, value in model.state_dict().items()
        },
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": (
            scaler.state_dict()
            if scaler is not None
            else None
        ),
        "history": history,
    }

    tmp = path.with_suffix(".tmp")
    torch.save(payload, tmp)
    tmp.replace(path)


def _save_final_seed(
    model,
    path,
    seed,
    trained_epochs,
):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    torch.save(
        {
            "architecture": "JointHurdleLSTM-v4-expanded-es",
            "training_signature": FINAL_TRAIN_SIGNATURE,
            "seed": int(seed),
            "trained_epochs": int(trained_epochs),
            "scheduler_t_max": int(MAX_CV_EPOCHS),
            "model_kwargs": MODEL_KWARGS,
            "state_dict": {
                key: value.detach().cpu()
                for key, value in model.state_dict().items()
            },
        },
        path,
    )


def _final_checkpoint_is_current(path):
    if not path.exists():
        return False

    try:
        checkpoint = _load_resume(
            path,
            map_location="cpu",
        )
    except Exception:
        return False

    return (
        checkpoint.get("architecture")
        == "JointHurdleLSTM-v4-expanded-es"
        and checkpoint.get("training_signature")
        == FINAL_TRAIN_SIGNATURE
        and int(
            checkpoint.get("trained_epochs", -1)
        )
        == int(BEST_EPOCH)
    )


if SHOULD_RUN_FINAL:
    for seed in FINAL_SEEDS:
        print("\n" + "=" * 100)
        print("FINAL SEED:", seed)

        seed_dir = (
            MODEL_DIR
            / f"seed_{seed}"
        )

        seed_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        resume_path = seed_dir / "resume.pt"
        final_path = seed_dir / "model.pt"
        history_path = seed_dir / "history.json"

        if _final_checkpoint_is_current(final_path):
            print(
                f"seed {seed}: current final model exists -> skip"
            )

            checkpoint = _load_resume(
                final_path,
                map_location=DEVICE,
            )

            model = JointHurdleLSTM(
                **checkpoint["model_kwargs"]
            ).to(DEVICE)

            model.load_state_dict(
                checkpoint["state_dict"],
                strict=True,
            )

            model.eval()

            history = (
                json.loads(
                    history_path.read_text(
                        encoding="utf-8"
                    )
                )
                if history_path.exists()
                else []
            )

            final_models.append(
                (seed, model, history)
            )

            continue

        if final_path.exists():
            print(
                f"seed {seed}: stale final model ignored"
            )

        set_seed(seed)

        train_loader = make_loader(
            LABELED_CUTOFFS,
            shuffle=True,
            known_user_mask=final_known_user_mask,
        )

        model = JointHurdleLSTM(
            **MODEL_KWARGS
        ).to(DEVICE)

        optimizer = make_optimizer(model)
        scaler = new_grad_scaler()

        # ВАЖНО: тот же schedule horizon, что и в CV.
        scheduler = (
            torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer,
                T_max=MAX_CV_EPOCHS,
                eta_min=2e-5,
            )
        )

        history = []
        start_epoch = 1

        if resume_path.exists():
            checkpoint = _load_resume(
                resume_path,
                map_location=DEVICE,
            )

            if (
                checkpoint.get("architecture")
                == "JointHurdleLSTM-v4-expanded-es"
                and checkpoint.get("training_signature")
                == FINAL_TRAIN_SIGNATURE
            ):
                model.load_state_dict(
                    checkpoint["state_dict"],
                    strict=True,
                )

                optimizer.load_state_dict(
                    checkpoint["optimizer_state_dict"]
                )

                scheduler.load_state_dict(
                    checkpoint["scheduler_state_dict"]
                )

                if (
                    scaler is not None
                    and checkpoint.get("scaler_state_dict")
                    is not None
                ):
                    scaler.load_state_dict(
                        checkpoint["scaler_state_dict"]
                    )

                history = checkpoint.get(
                    "history",
                    [],
                )

                start_epoch = (
                    int(checkpoint["epoch"])
                    + 1
                )

                print(
                    f"RESUME: epoch {checkpoint['epoch']} completed "
                    f"-> start from {start_epoch}"
                )

            else:
                print(
                    f"seed {seed}: stale resume checkpoint ignored"
                )

        for epoch in range(
            start_epoch,
            BEST_EPOCH + 1,
        ):
            train_metrics = train_one_epoch(
                model,
                train_loader,
                optimizer,
                final_static_stats,
                scaler,
            )

            scheduler.step()

            history.append({
                "epoch": int(epoch),
                "lr": float(
                    optimizer.param_groups[0]["lr"]
                ),
                **{
                    key: float(value)
                    for key, value
                    in train_metrics.items()
                },
            })

            _save_resume_checkpoint(
                resume_path,
                seed,
                epoch,
                model,
                optimizer,
                scheduler,
                scaler,
                history,
            )

            history_path.write_text(
                json.dumps(
                    history,
                    ensure_ascii=False,
                    indent=2,
                ),
                encoding="utf-8",
            )

            print(
                f"epoch {epoch:02d}/{BEST_EPOCH:02d} | "
                f"lr={optimizer.param_groups[0]['lr']:.2e} | "
                f"loss={train_metrics['loss']:.5f} | "
                f"main_RMSE="
                f"{np.sqrt(max(train_metrics['main_mse'], 0)):.5f} | "
                "checkpoint SAVED"
            )

        _save_final_seed(
            model,
            final_path,
            seed,
            BEST_EPOCH,
        )

        print(
            "FINAL WEIGHTS SAVED:",
            final_path,
        )

        final_models.append(
            (seed, model, history)
        )

else:
    print(
        "No final models trained."
    )

## Сохранение artifacts

Сохраняю:

- веса каждого final seed;
- primary `model.pt`;
- static mean/std;
- `meta.json`;
- список `all_user_ids`;
- CV history и summary;
- config с архитектурой и флагами.

Resume checkpoint каждого seed обновляется после каждой эпохи.

In [ ]:
def save_checkpoint(model, path, seed):
    path.parent.mkdir(parents=True, exist_ok=True)

    payload = {
        "architecture": "JointHurdleLSTM-v4-expanded-es",
        "training_signature": FINAL_TRAIN_SIGNATURE,
        "seed": int(seed),
        "trained_epochs": int(BEST_EPOCH),
        "scheduler_t_max": int(MAX_CV_EPOCHS),
        "model_kwargs": MODEL_KWARGS,
        "state_dict": {
            key: value.detach().cpu()
            for key, value in model.state_dict().items()
        },
    }

    torch.save(payload, path)


meta_bytes = (DATA_DIR / "meta.json").read_bytes()

cv_df.to_csv(MODEL_DIR / "cv_history.csv", index=False)
cv_summary.to_csv(MODEL_DIR / "cv_summary.csv", index=False)

shutil.copy2(DATA_DIR / "meta.json", MODEL_DIR / "data_meta.json")
shutil.copy2(DATA_DIR / "all_user_ids.npy", MODEL_DIR / "all_user_ids.npy")

if final_models:
    for seed, model, history in final_models:
        save_checkpoint(
            model,
            MODEL_DIR / f"seed_{seed}" / "model.pt",
            seed,
        )

    primary_seed, primary_model, _ = final_models[0]
    save_checkpoint(primary_model, MODEL_DIR / "model.pt", primary_seed)

    torch.save(
        {
            "mean": final_static_stats[0].cpu(),
            "std": final_static_stats[1].cpu(),
            "selected_static_features": STATIC_FEATURES,
        },
        MODEL_DIR / "static_stats.pt",
    )

config = {
    "architecture": "JointHurdleLSTM-v4-expanded-es",
    "data_format_version": META["format_version"],
    "data_meta_sha256": hashlib.sha256(meta_bytes).hexdigest(),
    "model_kwargs": MODEL_KWARGS,
    "static_features": STATIC_FEATURES,
    "flags": {
        "use_user_embedding": USE_USER_EMBEDDING,
        "use_conv_stem": USE_CONV_STEM,
        "use_absolute_time": USE_ABSOLUTE_TIME,
        "use_future_calendar_static": USE_FUTURE_CALENDAR_STATIC,
        "use_cutoff_seasonal_static": USE_CUTOFF_SEASONAL_STATIC,
    },
    "optimizer": "AdamW",
    "lr": LR,
    "embedding_lr": EMBED_LR,
    "weight_decay": WEIGHT_DECAY,
    "embedding_weight_decay": EMBED_WEIGHT_DECAY,
    "aux_weights": {
        "gate": AUX_GATE_WEIGHT,
        "positive": AUX_POS_WEIGHT,
        "direct": AUX_DIRECT_WEIGHT,
    },
    "cv_valid_cutoffs": CV_VALID_CUTOFFS,
    "best_epoch": BEST_EPOCH,
    "max_cv_epochs": MAX_CV_EPOCHS,
    "last_common_epoch": LAST_COMMON_EPOCH,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "early_stopping_min_delta": EARLY_STOPPING_MIN_DELTA,
    "min_cv_epochs_before_stop": MIN_CV_EPOCHS_BEFORE_STOP,
    "scheduler_t_max": MAX_CV_EPOCHS,
    "training_signature": FINAL_TRAIN_SIGNATURE,
    "mean_temporal_cv_rmsle": CV_RMSLE,
    "cv_std_rmsle": CV_STD,
    "january_rmsle": JAN_RMSLE,
    "reference_mean_cv_rmsle": REFERENCE_MEAN_CV_RMSLE,
    "reference_january_rmsle": REFERENCE_JAN_HOLDOUT_RMSLE,
    "screen_limit": jan_limit,
    "final_train_ran": bool(final_models),
    "final_seeds": FINAL_SEEDS if final_models else [],
    "prediction_space": "log1p",
    "final_ensemble": (
        "mean pred_log across seeds, then expm1"
        if final_models
        else None
    ),
    "sequence_input_size": SEQ_INPUT_SIZE,
    "short_summary_size": SHORT_SUMMARY_SIZE,
    "static_input_size": STATIC_INPUT_SIZE,
}

with open(MODEL_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("saved artifacts to:", MODEL_DIR)

## Inference и submission

Для каждого seed считаю `pred_log` на `2026-02-13`.

Финальный prediction:

`mean(pred_log_seed_1, pred_log_seed_2) -> expm1`

Проверяю:

- нет `NaN`;
- нет `inf`;
- нет отрицательных prediction;
- есть prediction для всех 250000 пользователей.

In [ ]:
if final_models:
    seed_pred_logs = []
    submission_user_ids = None

    for seed, model, _ in final_models:
        user_ids, pred, _ = predict_model(
            model,
            INFERENCE_CUTOFF,
            final_static_stats,
            final_known_user_mask,
            with_target=False,
        )
        seed_pred_logs.append(np.clip(pred["pred_log"], 0, None))
        submission_user_ids = user_ids

    ensemble_pred_log = np.mean(np.stack(seed_pred_logs, axis=0), axis=0)
    raw_pred = np.expm1(ensemble_pred_log)

    submission = pd.read_csv(PROJECT_ROOT / "data" / "sample_submit.csv")
    pred_map = pd.Series(raw_pred, index=submission_user_ids)
    submission["predict"] = submission["user_id"].map(pred_map)

    if submission["predict"].isna().any():
        raise AssertionError(
            f"Missing predictions: {int(submission['predict'].isna().sum())}"
        )
    if not np.isfinite(submission["predict"]).all():
        raise AssertionError("Non-finite predictions")
    if (submission["predict"] < 0).any():
        raise AssertionError("Negative predictions are forbidden")

    out_path = SUBMISSION_DIR / "lstm_hurdle_v4_expanded_es_4seed.csv"
    submission.to_csv(out_path, index=False)

    print("saved:", out_path)
    print("rows:", len(submission))
    print("exact zeros:", f"{(submission['predict'] == 0).mean():.2%}")
    display(submission.head())
else:
    print("Submission skipped because final training was skipped.")

## Reload smoke-test

После сохранения заново загружаю model + static stats и делаю forward на одном batch.

Проверяю конечность:

- `pred_log`;
- `gate_prob`;
- `positive_log`;
- `direct_log`.

In [ ]:
def safe_torch_load(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=map_location)


def load_saved_model(path):
    checkpoint = safe_torch_load(path, map_location=DEVICE)
    if checkpoint.get("architecture") != "JointHurdleLSTM-v4-expanded-es":
        raise RuntimeError("Unexpected checkpoint architecture")

    model = JointHurdleLSTM(**checkpoint["model_kwargs"]).to(DEVICE)
    model.load_state_dict(checkpoint["state_dict"], strict=True)
    model.eval()
    return model


if final_models:
    saved_stats = safe_torch_load(MODEL_DIR / "static_stats.pt", map_location="cpu")
    loaded_stats = (saved_stats["mean"], saved_stats["std"])
    loaded_model = load_saved_model(MODEL_DIR / "model.pt")

    smoke_dataset = HybridDataset(
        INFERENCE_CUTOFF,
        with_target=False,
        known_user_mask=final_known_user_mask,
    )
    smoke_loader = DataLoader(
        smoke_dataset,
        batch_size=min(256, BATCH_SIZE),
        shuffle=False,
        num_workers=0,
    )
    smoke_batch = next(iter(smoke_loader))

    sequence, static, user_index, user_known, history_length, _ = unpack_batch(
        smoke_batch,
        with_target=False,
    )
    static = normalize_static(static, loaded_stats)

    with torch.no_grad():
        out = loaded_model(
            sequence,
            static,
            user_index,
            user_known,
            history_length,
        )

    for key in ["pred_log", "gate_prob", "positive_log", "direct_log"]:
        assert torch.isfinite(out[key]).all(), key

    print("reload smoke-test passed:", len(out["pred_log"]), "rows")
    print("saved hurdle weight:", float(out["hurdle_weight"].cpu()))
    print("saved user scale:", float(out["user_scale"].cpu()))
else:
    print("Reload test skipped: no final checkpoint was trained.")

## Итог

Этот run отвечает сразу на две вещи:

1. где реально заканчивается полезное обучение LSTM по числу эпох;
2. дает ли более устойчивый four-seed ensemble дополнительный gain.

Training policy:

```text
temporal CV folds = 3
max epochs        = 30
early stopping    = patience 8
min delta         = 1e-4
```

`BEST_EPOCH` выбирается только среди common epochs всех трех fold.

После этого final train:

```text
all labeled data -> seed 42
all labeled data -> seed 143
all labeled data -> seed 67
all labeled data -> seed 2026
```

Все четыре модели имеют одну архитектуру и один `BEST_EPOCH`.

Final:

```text
mean pred_log across 4 seeds
-> expm1
-> lstm_hurdle_v4_expanded_es_4seed.csv
```

LightGBM oracle тиммейта пока выключен и на основной LSTM run не влияет.

Если `BEST_EPOCH == LAST_COMMON_EPOCH`, training budget снова уперся в границу.

Если `BEST_EPOCH < LAST_COMMON_EPOCH` и после него validation стабильно хуже -- по числу эпох эту архитектуру практически дожали.